In [73]:
import sys
from pathlib import Path
# sys.path.insert(0, str(Path(__file__).parent.parent))

from typing import Optional

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap

import nfl_data_py as nfl

import os
from sqlalchemy import (
    create_engine, Column, Integer, Float, String, Boolean,
    Date, DateTime, Text, ForeignKey, Index, UniqueConstraint, text,
inspect
)
from sqlalchemy.orm import DeclarativeBase, relationship, Session
from sqlalchemy.pool import StaticPool

from datetime import date

pd.set_option('display.max_columns', None)

In [74]:
current_path = os.getcwd()
DB_PATH = str(Path(current_path).parent.parent / "dynasty_scout.db")
DB_PATH

'/Users/sandeeptiwari/Desktop/dynasty-ai-engine/dynasty_scout.db'

# Feature Engineering

### Constants

In [17]:
# Peak age by position — after peak, age_vs_position_peak goes positive (declining)
POSITION_PEAK_AGE = {"QB": 34, "WR": 26, "RB": 24, "TE": 27}

# Injury status severity weights for injury designation count scoring
DESIGNATION_WEIGHT = {
    "Out": 1.0,
    "Doubtful": 0.8,
    "Questionable": 0.3,
    "Limited": 0.1,
    "Full": 0.0,
}

# Soft tissue and high-recurrence injuries
SOFT_TISSUE = {"Hamstring", "Quad", "Quadricep", "Groin", "Hip Flexor", "Calf"}
HIGH_SEVERITY = {"Knee", "ACL", "MCL", "Back", "Concussion"}

# Conference competition adjustment multiplier for college stats
CONFERENCE_MULTIPLIER = {1: 1.0, 2: 0.85, 3: 0.70}   # tier 1=P5, 2=G5, 3=FCS

## Loading Data

In [18]:
def get_engine(db_path: Path = DB_PATH, echo: bool = False):
    """
    Returns a SQLAlchemy engine. Uses StaticPool so the same connection
    is reused in single-threaded contexts (fine for local use).
    """
    return create_engine(
        f"sqlite:///{db_path}",
        connect_args={"check_same_thread": False},
        poolclass=StaticPool,
        echo=echo,
    )


def _load_nfl_stats(seasons: list[int]) -> pd.DataFrame:
        query = f"""
            SELECT s.*, p.position, p.nfl_team as current_team,
                   p.birth_date, p.years_exp, p.draft_round, p.draft_pick,
                   p.rookie_year, p.sleeper_id, p.name as player_name
            FROM nfl_season_stats s
            JOIN players p ON s.player_id = p.player_id
            WHERE s.season IN ({','.join(map(str, seasons))})
              AND s.season_type = 'REG'
              AND p.position IN ('QB', 'RB', 'WR', 'TE')
        """
        with engine.connect() as conn:
            return pd.read_sql(text(query), conn)

def _load_advanced_stats(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT * FROM nfl_advanced_stats
        WHERE season IN ({','.join(map(str, seasons))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

def _load_injury_data(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT * FROM injury_records
        WHERE season IN ({','.join(map(str, seasons + [s-1 for s in seasons]))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

def _load_snap_data(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT player_id, season, week, offense_pct
        FROM nfl_weekly_snaps
        WHERE season IN ({','.join(map(str, seasons))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

def _load_players() -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text("SELECT * FROM players"), conn)

def _load_team_context(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT * FROM nfl_teams
        WHERE season IN ({','.join(map(str, seasons))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

def _load_college_stats() -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text("SELECT * FROM college_season_stats"), conn)

def _load_combine_data() -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text("SELECT * FROM combine_measurements"), conn)

# custom functions
def load_nfl_season_stats(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT * FROM nfl_season_stats
        WHERE season IN ({','.join(map(str, seasons))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

def load_games_played() -> pd.DataFrame:
    query = f"""
        SELECT * FROM college_games_played
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

In [19]:
seasons = list(range(2015, 2025))
engine = get_engine()

In [20]:
def _merge_all(stats_df, snap_df, team_df) -> pd.DataFrame:
        """Merge all data sources onto the seasonal stats base."""
        df = stats_df.copy()

        # print("STATS", df.columns)
        # Snap pct trend (std dev of weekly snaps — measures role consistency)
        snap_consistency = (
            snap_df.groupby(["player_id", "season"])
            .agg(snap_pct_std=("offense_pct", "std"), snap_pct_mean=("offense_pct", "mean"))
            .reset_index()
        )
        snap_consistency.fillna({'snap_pct_std': 0}, inplace=True)
        df = df.merge(snap_consistency, on=["player_id", "season"], how="left")
        
        # Team context
        if not team_df.empty:
            print("TEAM", team_df.columns)
            df = df.merge(
                team_df.rename(columns={"team_abbr": "team"}),
                on=["team", "season"],
                how="left"
            )

        return df

In [21]:
stats_df = _load_nfl_stats(seasons)
adv_df = _load_advanced_stats(seasons)
injury_df = _load_injury_data(seasons)
snap_df = _load_snap_data(seasons)
players_df = _load_players()
team_df = _load_team_context(seasons)
college_df = _load_college_stats()
combine_df = _load_combine_data()

# custom data
nfl_season_stats_df = load_nfl_season_stats(seasons)
college_games_played = load_games_played()

In [66]:
injury_df

,id,player_id,season,week,team,report_status,practice_status,primary_injury,designation_weight
0,1,00-0039851,2024,7,NE,None,Full Participation in Practice,None,0.0
1,2,00-0039851,2024,8,NE,None,Full Participation in Practice,None,0.0
2,3,00-0039851,2024,9,NE,Questionable,Limited Participation in Practice,None,0.3
3,4,00-0039851,2024,18,NE,Questionable,Limited Participation in Practice,None,0.3
4,5,00-0039910,2024,8,WAS,Questionable,Limited Participation in Practice,None,0.3
...,...,...,...,...,...,...,...,...,...
44790,44791,00-0026512,2016,17,WAS,Questionable,Limited Participation in Practice,None,0.3
44791,44792,00-0026512,2017,3,TB,Doubtful,Did Not Participate In Practice,None,0.8
44792,44793,00-0026512,2017,4,TB,None,Full Participation in Practice,None,0.0
44793,44794,00-0026512,2017,8,TB,Questionable,Full Participation in Practice,None,0.3


In [67]:
injury_df['practice_status'].unique()

array(['Full Participation in Practice',
       'Limited Participation in Practice',
       'Did Not Participate In Practice', '\n    ', 'Note',
       'Out (Definitely Will Not Play)'], dtype=object)

In [68]:
injury_df['report_status'].unique()

array([None, 'Questionable', 'Out', 'Doubtful', 'Note', 'Probable'],
      dtype=object)

In [72]:
injury_df[injury_df['report_status'].isin(['Out', 'Doubtful', 'Questionable'])].groupby(["player_id", "season"]).size().reset_index(name="injury_designation_count")


,player_id,season,injury_designation_count
0,00-0004091,2018,3
1,00-0007091,2015,2
2,00-0010346,2015,6
3,00-0016919,2018,2
4,00-0016919,2019,1
...,...,...,...
8187,00-0039915,2024,5
8188,00-0039916,2024,1
8189,00-0039919,2024,1
8190,00-0039921,2024,3


In [8]:
ids = nfl.import_ids()
player_ids = ids[['gsis_id', 'pfr_id']][~ids['gsis_id'].isna()].rename(columns={'gsis_id': 'player_id', 'pfr_id': 'pfr_player_id'}).drop_duplicates(subset=['player_id'])


## Feature Engineering

### Generating Starting Dataset
___
This indexes on the data of the players for which we have NFL stats. I have created an alternative starting index that is much cleaner than the one already in place—we will use this moving forward

In [27]:
def load_joined_data(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT nfl.player_id,
               p.name,
               p.position,
               nfl.season,
               nfl.team,
               p.college_team, p.birth_date, p.status, p.height, p.weight, p.years_exp, p.draft_round, p.draft_pick, p.draft_team, p.rookie_year,
               
               nfl.games, nfl.completions, nfl.attempts, nfl.passing_yards, nfl.passing_tds, 
               nfl.interceptions, nfl.passing_epa, nfl.completion_pct, nfl.yards_per_attempt, 
               nfl.passer_rating, nfl.sacks, nfl.carries, nfl.rushing_yards, nfl.rushing_tds, 
               nfl.rushing_epa, nfl.yards_per_carry, nfl.targets, nfl.receptions, nfl.receiving_yards, 
               nfl.receiving_tds, nfl.receiving_epa, nfl.yards_per_reception, nfl.catch_rate, 
               nfl.yards_per_target, nfl.air_yards_total, nfl.yards_after_catch, nfl.fantasy_points_ppr, 
               nfl.fantasy_ppg_ppr, nfl.snap_pct, nfl.target_share, nfl.air_yards_share, nfl.racr, nfl.wopr, nfl.tgt_per_game
        FROM nfl_season_stats as nfl
        LEFT JOIN players p
        ON nfl.player_id = p.player_id
        WHERE name is not NULL
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

In [28]:
join_df = load_joined_data(seasons)
join_df

,player_id,name,position,season,team,college_team,birth_date,status,height,weight,years_exp,draft_round,draft_pick,draft_team,rookie_year,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,carries,rushing_yards,rushing_tds,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_ppg_ppr,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game
0,00-0007091,Matt Hasselbeck,QB,2015,IND,Boston College,1975-09-25,ACT,76,235,17,6.0,NaN,GB,1999,8,156,256,1690,9,5,-0.113061,0.609375,6.601562,83.951823,16,16,15,0,-5.870652,0.937500,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.10,11.387500,0.867500,0.000000,0.000000,0.000000,0.000000,0.000000
1,00-0010346,Peyton Manning,QB,2015,CAR,Tennessee,1976-03-24,ACT,77,230,18,1.0,NaN,IND,1998,10,198,331,2249,9,17,-31.956645,0.598187,6.794562,67.906596,16,6,-6,0,-3.616597,-1.000000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.36,9.136000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,00-0019596,Tom Brady,QB,2015,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,402,624,4770,36,7,127.536179,0.644231,7.644231,102.176816,38,34,53,3,6.768190,1.558824,1,1,36,0,3.129031,36.000000,1.000000,36.000000,7,29,344.70,21.543750,0.989444,0.017857,0.012259,5.142857,0.035367,0.062500
3,00-0019596,Tom Brady,QB,2016,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,12,291,432,3554,28,2,147.296480,0.673611,8.226852,112.172068,15,28,64,0,-5.717147,2.285714,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,258.56,21.546667,0.972667,0.000000,0.000000,0.000000,0.000000,0.000000
4,00-0019596,Tom Brady,QB,2017,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,385,581,4577,32,8,138.508184,0.662651,7.877797,102.750287,35,25,28,0,-12.203785,1.120000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,295.88,18.492500,0.980526,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5582,00-0039917,Michael Penix Jr.,QB,2024,ATL,Washington,2000-05-08,ACT,75,220,3,1.0,8.0,ATL,2024,5,61,105,775,3,3,14.553117,0.580952,7.380952,78.869048,4,7,11,1,0.585619,1.571429,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,44.10,8.820000,0.664000,0.000000,0.000000,0.000000,0.000000,0.000000
5583,00-0039918,Caleb Williams,QB,2024,CHI,USC,2001-11-18,ACT,73,226,3,1.0,1.0,CHI,2024,17,351,562,3541,20,6,-43.804843,0.624555,6.300712,87.796560,68,81,489,0,11.632084,6.037037,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,254.54,14.972941,0.987647,0.000000,0.000000,0.000000,0.000000,0.000000
5584,00-0039919,Rome Odunze,WR,2024,CHI,Washington,2002-06-03,ACT,75,215,3,1.0,9.0,CHI,2024,17,0,0,0,0,0,0.000000,NaN,NaN,NaN,0,3,15,0,0.575859,5.000000,101,54,734,3,16.061161,13.592593,0.534653,7.267327,1398,253,144.90,8.523529,0.835882,3.215179,5.542503,10.960008,8.702520,5.941176
5585,00-0039920,Malachi Corley,WR,2024,NYJ,Western Kentucky,2002-03-21,ACT,71,215,3,3.0,65.0,NYJ,2024,5,0,0,0,0,0,0.000000,NaN,NaN,NaN,0,2,26,0,-4.431244,13.000000,6,3,16,0,-3.643815,5.333333,0.500000,2.666667,50,5,7.20,1.440000,0.151111,0.182203,0.225307,1.166667,0.431020,1.200000


In [29]:
missing_pct = (
    join_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                feature  missing_pct
0         passer_rating     0.830857
1     yards_per_attempt     0.830857
2        completion_pct     0.830857
3       yards_per_carry     0.431359
4            draft_pick     0.298729
5           draft_round     0.292465
6            draft_team     0.292465
7   yards_per_reception     0.176660
8      yards_per_target     0.145516
9            catch_rate     0.145516
10                 team     0.014498
11             snap_pct     0.014498
12      air_yards_share     0.000000
13          rushing_tds     0.000000
14      air_yards_total     0.000000
15         target_share     0.000000
16    yards_after_catch     0.000000
17      fantasy_ppg_ppr     0.000000
18        receiving_epa     0.000000
19        receiving_tds     0.000000
20      receiving_yards     0.000000
21           receptions     0.000000
22   fantasy_points_ppr     0.000000
23                 racr     0.000000
24              targets     0.000000
25                 wopr     0.000000
2

In [30]:
join_df = _merge_all(join_df, snap_df, team_df)
join_df

TEAM Index(['team_abbr', 'season', 'full_name', 'head_coach',
       'offensive_coordinator', 'defensive_coordinator', 'offensive_scheme',
       'plays_per_game', 'pass_rate', 'pass_rate_neutral', 'team_pass_yards',
       'team_rush_yards', 'team_total_tds', 'team_pass_attempts',
       'team_targets', 'points_per_game', 'offensive_line_rank'],
      dtype='object')


,player_id,name,position,season,team,college_team,birth_date,status,height,weight,years_exp,draft_round,draft_pick,draft_team,rookie_year,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,carries,rushing_yards,rushing_tds,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_ppg_ppr,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game,snap_pct_std,snap_pct_mean,full_name,head_coach,offensive_coordinator,defensive_coordinator,offensive_scheme,plays_per_game,pass_rate,pass_rate_neutral,team_pass_yards,team_rush_yards,team_total_tds,team_pass_attempts,team_targets,points_per_game,offensive_line_rank
0,00-0007091,Matt Hasselbeck,QB,2015,IND,Boston College,1975-09-25,ACT,76,235,17,6.0,NaN,GB,1999,8,156,256,1690,9,5,-0.113061,0.609375,6.601562,83.951823,16,16,15,0,-5.870652,0.937500,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.10,11.387500,0.867500,0.000000,0.000000,0.000000,0.000000,0.000000,0.197032,0.867500,Indianapolis Colts,Chuck Pagano,Rob Chudzinski,Greg Manusky,None,65.88,0.6243,0.5876,3928.0,1438.0,None,658.0,621.0,20.81,15.0
1,00-0010346,Peyton Manning,QB,2015,CAR,Tennessee,1976-03-24,ACT,77,230,18,1.0,NaN,IND,1998,10,198,331,2249,9,17,-31.956645,0.598187,6.794562,67.906596,16,6,-6,0,-3.616597,-1.000000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.36,9.136000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.208431,0.914615,Carolina Panthers,Ron Rivera,Mike Shula,Sean McDermott,West Coast,66.63,0.5016,0.5259,4634.0,2696.0,None,635.0,593.0,31.25,22.0
2,00-0019596,Tom Brady,QB,2015,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,402,624,4770,36,7,127.536179,0.644231,7.644231,102.176816,38,34,53,3,6.768190,1.558824,1,1,36,0,3.129031,36.000000,1.000000,36.000000,7,29,344.70,21.543750,0.989444,0.017857,0.012259,5.142857,0.035367,0.062500,0.021549,0.989444,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,65.78,0.6503,0.6693,5424.0,1486.0,None,770.0,728.0,29.06,11.0
3,00-0019596,Tom Brady,QB,2016,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,12,291,432,3554,28,2,147.296480,0.673611,8.226852,112.172068,15,28,64,0,-5.717147,2.285714,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,258.56,21.546667,0.972667,0.000000,0.000000,0.000000,0.000000,0.000000,0.057379,0.972667,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.84,0.5640,0.5969,5593.0,2131.0,None,727.0,694.0,27.56,6.0
4,00-0019596,Tom Brady,QB,2017,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,385,581,4577,32,8,138.508184,0.662651,7.877797,102.750287,35,25,28,0,-12.203785,1.120000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,295.88,18.492500,0.980526,0.000000,0.000000,0.000000,0.000000,0.000000,0.034877,0.980526,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.63,0.5984,0.6057,5771.0,2149.0,None,769.0,730.0,28.62,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5582,00-0039917,Michael Penix Jr.,QB,2024,ATL,Washington,2000-05-08,ACT,75,220,3,1.0,8.0,ATL,2024,5,61,105,775,3,3,14.553117,0.580952,7.380952,78.869048,4,7,11,1,0.585619,1.571429,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,44.10,8.820000,0.664000,0.000000,0.000000,0.000000,0.000000,0.000000,0.462039,0.664000,Atlanta Falcons,Raheem Morris,Zac Robinson,Jimmy Lake,Air Coryell,64.12,0.5440,0.5314,4283.0,2219.0,None,593.0,561.0,22.88,6.0
5583,00-0039918,Caleb Williams,QB,2024,CHI,USC,2001-11-18,ACT,73,226,3,1.0,1.0,CHI,2024,17,351,562,3541,20,6,-43.804843,0.624555,6.300712,87.796560,68,81,489,0,11.632084,6.037037,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,254.54,14.972941,0.987647,0.000000,0.000000,0

In [31]:
missing_pct = (
    join_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                  feature  missing_pct
0          team_total_tds     1.000000
1           passer_rating     0.830857
2       yards_per_attempt     0.830857
3          completion_pct     0.830857
4         yards_per_carry     0.431359
5              draft_pick     0.298729
6             draft_round     0.292465
7              draft_team     0.292465
8     yards_per_reception     0.176660
9              catch_rate     0.145516
10       yards_per_target     0.145516
11  defensive_coordinator     0.104707
12  offensive_coordinator     0.065330
13       offensive_scheme     0.041346
14             head_coach     0.037945
15    offensive_line_rank     0.037945
16         plays_per_game     0.037945
17        team_rush_yards     0.037945
18        points_per_game     0.037945
19           team_targets     0.037945
20              pass_rate     0.037945
21     team_pass_attempts     0.037945
22              full_name     0.037945
23        team_pass_yards     0.037945
24      pass_rate_neutral

### Add Performance Trajectory

In [32]:
def _add_performance_trajectory(df: pd.DataFrame) -> pd.DataFrame:
    """
    Multi-year fantasy PPG trends.
    Rolling averages and slope of recent performance.
    """
    df = df.sort_values(["player_id", "season"])

    # Rolling averages (shift to avoid leakage — only use past seasons)
    grp = df.groupby("player_id")["fantasy_ppg_ppr"]
    df["fantasy_ppg_last_season"] = grp.shift(1)
    df["fantasy_ppg_2yr_avg"] = grp.shift(1).rolling(2, min_periods=1).mean().values
    df["fantasy_ppg_3yr_avg"] = grp.shift(1).rolling(3, min_periods=1).mean().values

    # Trend: slope of last 3 seasons (positive = improving, negative = declining)
    def rolling_slope(series: pd.Series, n: int = 3) -> pd.Series:
        """OLS slope over last n seasons."""
        slopes = []
        vals = series.tolist()
        for i in range(len(vals)):
            window = [v for v in vals[max(0, i-n):i] if pd.notna(v)]
            if len(window) >= 2:
                x = np.arange(len(window), dtype=float)
                y = np.array(window, dtype=float)
                slope = np.polyfit(x, y, 1)[0]
                slopes.append(slope)
            else:
                slopes.append(np.nan)
        return pd.Series(slopes, index=series.index)

    df["fantasy_ppg_trend"] = (
        df.groupby("player_id")["fantasy_ppg_ppr"]
        .transform(lambda s: rolling_slope(s))
    )

    # Career games (context for sample size)
    df["career_games"] = df.groupby("player_id")["games"].transform(
        lambda s: pd.to_numeric(s, errors="coerce").fillna(0).cumsum().shift(1)
    )

    return df

In [33]:
join_df = _add_performance_trajectory(join_df)
join_df

,player_id,name,position,season,team,college_team,birth_date,status,height,weight,years_exp,draft_round,draft_pick,draft_team,rookie_year,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,carries,rushing_yards,rushing_tds,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_ppg_ppr,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game,snap_pct_std,snap_pct_mean,full_name,head_coach,offensive_coordinator,defensive_coordinator,offensive_scheme,plays_per_game,pass_rate,pass_rate_neutral,team_pass_yards,team_rush_yards,team_total_tds,team_pass_attempts,team_targets,points_per_game,offensive_line_rank,fantasy_ppg_last_season,fantasy_ppg_2yr_avg,fantasy_ppg_3yr_avg,fantasy_ppg_trend,career_games
0,00-0007091,Matt Hasselbeck,QB,2015,IND,Boston College,1975-09-25,ACT,76,235,17,6.0,NaN,GB,1999,8,156,256,1690,9,5,-0.113061,0.609375,6.601562,83.951823,16,16,15,0,-5.870652,0.937500,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.10,11.387500,0.867500,0.000000,0.000000,0.000000,0.000000,0.000000,0.197032,0.867500,Indianapolis Colts,Chuck Pagano,Rob Chudzinski,Greg Manusky,None,65.88,0.6243,0.5876,3928.0,1438.0,None,658.0,621.0,20.81,15.0,NaN,NaN,NaN,NaN,NaN
1,00-0010346,Peyton Manning,QB,2015,CAR,Tennessee,1976-03-24,ACT,77,230,18,1.0,NaN,IND,1998,10,198,331,2249,9,17,-31.956645,0.598187,6.794562,67.906596,16,6,-6,0,-3.616597,-1.000000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.36,9.136000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.208431,0.914615,Carolina Panthers,Ron Rivera,Mike Shula,Sean McDermott,West Coast,66.63,0.5016,0.5259,4634.0,2696.0,None,635.0,593.0,31.25,22.0,NaN,NaN,NaN,NaN,NaN
2,00-0019596,Tom Brady,QB,2015,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,402,624,4770,36,7,127.536179,0.644231,7.644231,102.176816,38,34,53,3,6.768190,1.558824,1,1,36,0,3.129031,36.000000,1.000000,36.000000,7,29,344.70,21.543750,0.989444,0.017857,0.012259,5.142857,0.035367,0.062500,0.021549,0.989444,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,65.78,0.6503,0.6693,5424.0,1486.0,None,770.0,728.0,29.06,11.0,NaN,NaN,NaN,NaN,NaN
3,00-0019596,Tom Brady,QB,2016,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,12,291,432,3554,28,2,147.296480,0.673611,8.226852,112.172068,15,28,64,0,-5.717147,2.285714,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,258.56,21.546667,0.972667,0.000000,0.000000,0.000000,0.000000,0.000000,0.057379,0.972667,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.84,0.5640,0.5969,5593.0,2131.0,None,727.0,694.0,27.56,6.0,21.543750,21.543750,21.543750,NaN,16.0
4,00-0019596,Tom Brady,QB,2017,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,385,581,4577,32,8,138.508184,0.662651,7.877797,102.750287,35,25,28,0,-12.203785,1.120000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,295.88,18.492500,0.980526,0.000000,0.000000,0.000000,0.000000,0.000000,0.034877,0.980526,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.63,0.5984,0.6057,5771.0,2149.0,None,769.0,730.0,28.62,9.0,21.546667,21.545208,21.545208,0.002917,28.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5582,00-0039917,Michael Penix Jr.,QB,2024,ATL,Washington,2000-05-08,ACT,75,220,3,1.0,8.0,ATL,2024,5,61,105,775,3,3,14.553117,0.580952,7.380952,78.869048,4,7,11,1,0.585619,1.571429,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,44.10,8.820000,0.664000,0.000000,0.000000,0.000000,0.000000,0.000000,0.462039,0.664000,Atlanta Falcons,Raheem Morris,Zac Robinson,Jimmy Lake,Air Coryell,64.12,0.5440,0.5314,4283.0,2219.0,None,593.0,561.0,22.88,6.0

In [35]:
missing_pct = (
    join_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                    feature  missing_pct
0            team_total_tds     1.000000
1            completion_pct     0.830857
2             passer_rating     0.830857
3         yards_per_attempt     0.830857
4         fantasy_ppg_trend     0.513514
5           yards_per_carry     0.431359
6   fantasy_ppg_last_season     0.298729
7                draft_pick     0.298729
8              career_games     0.298729
9               draft_round     0.292465
10               draft_team     0.292465
11      yards_per_reception     0.176660
12               catch_rate     0.145516
13         yards_per_target     0.145516
14    defensive_coordinator     0.104707
15      fantasy_ppg_2yr_avg     0.083945
16    offensive_coordinator     0.065330
17         offensive_scheme     0.041346
18                pass_rate     0.037945
19      offensive_line_rank     0.037945
20        pass_rate_neutral     0.037945
21          team_pass_yards     0.037945
22          team_rush_yards     0.037945
23       team_pa

### Add Role Features

In [36]:
def rolling_diff(series: pd.Series) -> pd.Series:
    """Year-over-year change in a metric."""
    return series.diff()

def _add_role_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Target share, air yards share, snap %, carries per game.
    Role stability is crucial — a high-usage player with declining team
    pass rate is less valuable than the raw numbers suggest.
    """
    df["targets_per_game"] = df["targets"] / df["games"].replace(0, np.nan)
    df["carries_per_game"] = df["carries"] / df["games"].replace(0, np.nan)

    # Snap % trend (positive = growing role)
    df = df.sort_values(["player_id", "season"])
    df["snap_pct_trend"] = (
        df.groupby("player_id")["snap_pct"]
        .transform(lambda s: rolling_diff(pd.to_numeric(s, errors="coerce")))
    )

    # Role security score: composite of snap_pct + target_share + carries_per_game (position-adjusted)
    df["role_security_score"] = np.where(
        df["position"].isin(["WR", "TE"]),
        0.4 * df["snap_pct"].fillna(0) + 0.6 * df["target_share"].fillna(0) * 10,
        0.5 * df["snap_pct"].fillna(0) + 0.5 * (df["carries_per_game"].fillna(0) / 20),
    )

    return df

In [37]:
join_df = _add_role_features(join_df)
join_df

,player_id,name,position,season,team,college_team,birth_date,status,height,weight,years_exp,draft_round,draft_pick,draft_team,rookie_year,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,carries,rushing_yards,rushing_tds,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_ppg_ppr,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game,snap_pct_std,snap_pct_mean,full_name,head_coach,offensive_coordinator,defensive_coordinator,offensive_scheme,plays_per_game,pass_rate,pass_rate_neutral,team_pass_yards,team_rush_yards,team_total_tds,team_pass_attempts,team_targets,points_per_game,offensive_line_rank,fantasy_ppg_last_season,fantasy_ppg_2yr_avg,fantasy_ppg_3yr_avg,fantasy_ppg_trend,career_games,targets_per_game,carries_per_game,snap_pct_trend,role_security_score
0,00-0007091,Matt Hasselbeck,QB,2015,IND,Boston College,1975-09-25,ACT,76,235,17,6.0,NaN,GB,1999,8,156,256,1690,9,5,-0.113061,0.609375,6.601562,83.951823,16,16,15,0,-5.870652,0.937500,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.10,11.387500,0.867500,0.000000,0.000000,0.000000,0.000000,0.000000,0.197032,0.867500,Indianapolis Colts,Chuck Pagano,Rob Chudzinski,Greg Manusky,None,65.88,0.6243,0.5876,3928.0,1438.0,None,658.0,621.0,20.81,15.0,NaN,NaN,NaN,NaN,NaN,0.000000,2.000000,NaN,0.483750
1,00-0010346,Peyton Manning,QB,2015,CAR,Tennessee,1976-03-24,ACT,77,230,18,1.0,NaN,IND,1998,10,198,331,2249,9,17,-31.956645,0.598187,6.794562,67.906596,16,6,-6,0,-3.616597,-1.000000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.36,9.136000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.208431,0.914615,Carolina Panthers,Ron Rivera,Mike Shula,Sean McDermott,West Coast,66.63,0.5016,0.5259,4634.0,2696.0,None,635.0,593.0,31.25,22.0,NaN,NaN,NaN,NaN,NaN,0.000000,0.600000,NaN,0.515000
2,00-0019596,Tom Brady,QB,2015,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,402,624,4770,36,7,127.536179,0.644231,7.644231,102.176816,38,34,53,3,6.768190,1.558824,1,1,36,0,3.129031,36.000000,1.000000,36.000000,7,29,344.70,21.543750,0.989444,0.017857,0.012259,5.142857,0.035367,0.062500,0.021549,0.989444,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,65.78,0.6503,0.6693,5424.0,1486.0,None,770.0,728.0,29.06,11.0,NaN,NaN,NaN,NaN,NaN,0.062500,2.125000,NaN,0.547847
3,00-0019596,Tom Brady,QB,2016,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,12,291,432,3554,28,2,147.296480,0.673611,8.226852,112.172068,15,28,64,0,-5.717147,2.285714,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,258.56,21.546667,0.972667,0.000000,0.000000,0.000000,0.000000,0.000000,0.057379,0.972667,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.84,0.5640,0.5969,5593.0,2131.0,None,727.0,694.0,27.56,6.0,21.543750,21.543750,21.543750,NaN,16.0,0.000000,2.333333,-0.016778,0.544667
4,00-0019596,Tom Brady,QB,2017,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,385,581,4577,32,8,138.508184,0.662651,7.877797,102.750287,35,25,28,0,-12.203785,1.120000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,295.88,18.492500,0.980526,0.000000,0.000000,0.000000,0.000000,0.000000,0.034877,0.980526,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.63,0.5984,0.6057,5771.0,2149.0,None,769.0,730.0,28.62,9.0,21.546667,21.545208,21.545208,0.002917,28.0,0.000000,1.562500,0.007860,0.529326
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5582,00-0039917,Michael Penix Jr.,QB,2024,ATL,Washington,2000-05-08,ACT,75,220,3,1.0,8.0,ATL,2024,5,61,105,775,3,3,14.553117,0.580952,7.380952,78.869048,4,7,11,1,0.585619,1.57

In [39]:
missing_pct = (
    join_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                    feature  missing_pct
0            team_total_tds     1.000000
1            completion_pct     0.830857
2         yards_per_attempt     0.830857
3             passer_rating     0.830857
4         fantasy_ppg_trend     0.513514
5           yards_per_carry     0.431359
6            snap_pct_trend     0.309289
7              career_games     0.298729
8                draft_pick     0.298729
9   fantasy_ppg_last_season     0.298729
10               draft_team     0.292465
11              draft_round     0.292465
12      yards_per_reception     0.176660
13         yards_per_target     0.145516
14               catch_rate     0.145516
15    defensive_coordinator     0.104707
16      fantasy_ppg_2yr_avg     0.083945
17    offensive_coordinator     0.065330
18         offensive_scheme     0.041346
19               head_coach     0.037945
20                full_name     0.037945
21           plays_per_game     0.037945
22                pass_rate     0.037945
23        pass_r

### Efficiency Features

In [40]:
def _add_efficiency_features(df: pd.DataFrame, adv_df: pd.DataFrame) -> pd.DataFrame:
    """
    EPA, CPOE, RYOE — these measure how much better/worse a player performs
    than average given the same opportunities. Critical for identifying
    true talent vs volume.
    """
    # EPA per play (passing)
    df["passing_epa_per_att"] = df["passing_epa"] / df["attempts"].replace(0, np.nan)

    # EPA per rush
    df["rushing_epa_per_carry"] = df["rushing_epa"] / df["carries"].replace(0, np.nan)

    # EPA per target (receiving)
    df["receiving_epa_per_tgt"] = df["receiving_epa"] / df["targets"].replace(0, np.nan)

    # RACR: Receiver Air Conversion Ratio = receiving_yards / air_yards
    # > 1.0 means player gains more than their air yards (good YAC)
    df["racr"] = df["receiving_yards"] / df["air_yards_total"].replace(0, np.nan)

    # YAC per reception
    df["yac_per_rec"] = df["yards_after_catch"] / df["receptions"].replace(0, np.nan)

    # Merge in NGS metrics (CPOE, RYOE, separation)
    passing_ngs = adv_df[adv_df["stat_type"] == "passing"][
        ["player_id", "season", "completion_pct_above_expectation",
         "avg_time_to_throw", "aggressiveness", "avg_intended_air_yards"]
    ].rename(columns={"completion_pct_above_expectation": "cpoe"})

    rushing_ngs = adv_df[adv_df["stat_type"] == "rushing"][
        ["player_id", "season", "rush_yards_over_expected_per_att",
         "efficiency", "avg_time_to_los"]
    ].rename(columns={"rush_yards_over_expected_per_att": "ryoe_per_att"})

    receiving_ngs = adv_df[adv_df["stat_type"] == "receiving"][
        ["player_id", "season", "avg_separation", "avg_cushion",
         "catch_pct_above_expectation", "avg_yac_above_expectation"]
    ]

    df = df.merge(passing_ngs, on=["player_id", "season"], how="left")
    df = df.merge(rushing_ngs, on=["player_id", "season"], how="left")
    df = df.merge(receiving_ngs, on=["player_id", "season"], how="left")

    return df

In [41]:
join_df = _add_efficiency_features(join_df, adv_df)
join_df

,player_id,name,position,season,team,college_team,birth_date,status,height,weight,years_exp,draft_round,draft_pick,draft_team,rookie_year,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,carries,rushing_yards,rushing_tds,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_ppg_ppr,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game,snap_pct_std,snap_pct_mean,full_name,head_coach,offensive_coordinator,defensive_coordinator,offensive_scheme,plays_per_game,pass_rate,pass_rate_neutral,team_pass_yards,team_rush_yards,team_total_tds,team_pass_attempts,team_targets,points_per_game,offensive_line_rank,fantasy_ppg_last_season,fantasy_ppg_2yr_avg,fantasy_ppg_3yr_avg,fantasy_ppg_trend,career_games,targets_per_game,carries_per_game,snap_pct_trend,role_security_score,passing_epa_per_att,rushing_epa_per_carry,receiving_epa_per_tgt,yac_per_rec,cpoe,avg_time_to_throw,aggressiveness,avg_intended_air_yards,ryoe_per_att,efficiency,avg_time_to_los,avg_separation,avg_cushion,catch_pct_above_expectation,avg_yac_above_expectation
0,00-0007091,Matt Hasselbeck,QB,2015,IND,Boston College,1975-09-25,ACT,76,235,17,6.0,NaN,GB,1999,8,156,256,1690,9,5,-0.113061,0.609375,6.601562,83.951823,16,16,15,0,-5.870652,0.937500,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.10,11.387500,0.867500,0.000000,0.000000,NaN,0.000000,0.000000,0.197032,0.867500,Indianapolis Colts,Chuck Pagano,Rob Chudzinski,Greg Manusky,None,65.88,0.6243,0.5876,3928.0,1438.0,None,658.0,621.0,20.81,15.0,NaN,NaN,NaN,NaN,NaN,0.000000,2.000000,NaN,0.483750,-0.000442,-0.366916,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00-0010346,Peyton Manning,QB,2015,CAR,Tennessee,1976-03-24,ACT,77,230,18,1.0,NaN,IND,1998,10,198,331,2249,9,17,-31.956645,0.598187,6.794562,67.906596,16,6,-6,0,-3.616597,-1.000000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.36,9.136000,1.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.208431,0.914615,Carolina Panthers,Ron Rivera,Mike Shula,Sean McDermott,West Coast,66.63,0.5016,0.5259,4634.0,2696.0,None,635.0,593.0,31.25,22.0,NaN,NaN,NaN,NaN,NaN,0.000000,0.600000,NaN,0.515000,-0.096546,-0.602766,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00-0019596,Tom Brady,QB,2015,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,402,624,4770,36,7,127.536179,0.644231,7.644231,102.176816,38,34,53,3,6.768190,1.558824,1,1,36,0,3.129031,36.000000,1.000000,36.000000,7,29,344.70,21.543750,0.989444,0.017857,0.012259,5.142857,0.035367,0.062500,0.021549,0.989444,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,65.78,0.6503,0.6693,5424.0,1486.0,None,770.0,728.0,29.06,11.0,NaN,NaN,NaN,NaN,NaN,0.062500,2.125000,NaN,0.547847,0.204385,0.199064,3.129031,29.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00-0019596,Tom Brady,QB,2016,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,12,291,432,3554,28,2,147.296480,0.673611,8.226852,112.172068,15,28,64,0,-5.717147,2.285714,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,258.56,21.546667,0.972667,0.000000,0.000000,NaN,0.000000,0.000000,0.057379,0.972667,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.84,0.5640,0.5969,5593.0,2131.0,None,727.0,694.0,27.56,6.0,21.543750,21.543750,21.543750,NaN,16.0,0.000000,2.333333,-0.016778,0.544667,0.340964,-0.204184,NaN,NaN,3.269891,2.559618,17.592593,8.112685,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00-0019596,Tom Brady,QB,2017,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,385,581,4577,32,8,138.508184,0.662651,7.877797,102.750287,35,25,28,0,-12.203785,1.120000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,295.88,18.492500,0.980526,0.000000,0.000000,NaN,0.000000,0.000000,0.034877,0.980526,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.63,0.5984,0.6057,5771.0,2149.0,None,769.0,730.0,28.62,9.0,21.5

In [43]:
missing_pct = (
    join_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                        feature  missing_pct
0   catch_pct_above_expectation     1.000000
1                team_total_tds     1.000000
2                  ryoe_per_att     0.936639
3                aggressiveness     0.934670
4             avg_time_to_throw     0.934670
5        avg_intended_air_yards     0.934670
6                          cpoe     0.934670
7               avg_time_to_los     0.918203
8                    efficiency     0.918203
9           passing_epa_per_att     0.830857
10            yards_per_attempt     0.830857
11                passer_rating     0.830857
12               completion_pct     0.830857
13               avg_separation     0.797566
14                  avg_cushion     0.797566
15    avg_yac_above_expectation     0.797566
16            fantasy_ppg_trend     0.513514
17              yards_per_carry     0.431359
18        rushing_epa_per_carry     0.431359
19               snap_pct_trend     0.309289
20      fantasy_ppg_last_season     0.298729
21        

### Injury Risk Features

In [44]:
def _add_injury_features(df: pd.DataFrame, injury_df: pd.DataFrame) -> pd.DataFrame:
    """
    Builds injury risk features from weekly injury report data.
    Key insight: injury TYPE matters more than injury count.
    - Soft tissue injuries (hamstring, groin) have high recurrence
    - ACL is a career trajectory disruptor but low recurrence after full recovery
    - Concussions accumulate risk

    The composite injury_risk_score is used as a feature in the performance
    forecaster AND as a standalone prediction by the injury risk model.
    """
    if injury_df.empty:
        df["injury_risk_score"] = np.nan
        return df

    # Games missed = games with "Out" or "Doubtful" designation
    games_missed = (
        injury_df[injury_df["report_status"].isin(["Out", "Doubtful"])]
        .groupby(["player_id", "season"])
        .size()
        .reset_index(name="games_missed")
    )

    # Weighted injury designation count (Out=1.0, Questionable=0.3, etc.)
    injury_df["designation_weight"] = injury_df["report_status"].map(
        DESIGNATION_WEIGHT
    ).fillna(0)

    weighted_designations = (
        injury_df.groupby(["player_id", "season"])["designation_weight"]
        .sum()
        .reset_index(name="weighted_injury_score")
    )

    # Injury type flags
    soft_tissue_flag = (
        injury_df[injury_df["primary_injury"].isin(SOFT_TISSUE)]
        .groupby(["player_id", "season"])
        .size()
        .reset_index(name="soft_tissue_count")
    )
    soft_tissue_flag["soft_tissue_injury_flag"] = True

    acl_flag = (
        injury_df[injury_df["primary_injury"].str.contains("ACL|Knee", na=False)]
        .groupby(["player_id", "season"])
        .size()
        .reset_index(name="acl_count")
    )
    acl_flag["acl_history_flag"] = True

    concussion_count = (
        injury_df[injury_df["primary_injury"].str.contains("Concussion", na=False)]
        .groupby(["player_id", "season"])
        .size()
        .reset_index(name="concussion_history_count")
    )

    # 2-year rolling games missed (injury track record)
    games_missed_2yr = (
        games_missed.sort_values(["player_id", "season"])
        .groupby("player_id")
        .apply(lambda g: g.assign(
            games_missed_2yr=g["games_missed"].rolling(2, min_periods=1).sum().shift(1)
        ))
        .reset_index(drop=True)
    )

    # Merge all injury features into main df
    df = df.merge(games_missed.rename(columns={"games_missed": "games_missed_last_season"}),
                  on=["player_id", "season"], how="left")
    df = df.merge(games_missed_2yr[["player_id", "season", "games_missed_2yr"]].rename(
                  columns={"games_missed_2yr": "games_missed_2yr_total"}),
                  on=["player_id", "season"], how="left")
    df = df.merge(weighted_designations, on=["player_id", "season"], how="left")
    df = df.merge(soft_tissue_flag[["player_id", "season", "soft_tissue_injury_flag"]],
                  on=["player_id", "season"], how="left")
    df = df.merge(acl_flag[["player_id", "season", "acl_history_flag"]],
                  on=["player_id", "season"], how="left")
    df = df.merge(concussion_count, on=["player_id", "season"], how="left")

    # Fill NaN flags with False/0
    df["soft_tissue_injury_flag"] = df["soft_tissue_injury_flag"].fillna(False)
    df["acl_history_flag"] = df["acl_history_flag"].fillna(False)
    df["games_missed_last_season"] = df["games_missed_last_season"].fillna(0)
    df["games_missed_2yr_total"] = df["games_missed_2yr_total"].fillna(0)
    df["concussion_history_count"] = df["concussion_history_count"].fillna(0)

    # Composite injury risk score (0-1)
    # Weighted combination of: games missed rate, soft tissue history, ACL history, concussions
    games_in_season = 17
    df["injury_risk_score"] = (
        0.35 * (df["games_missed_last_season"] / games_in_season).clip(0, 1)  # recency
        + 0.25 * (df["games_missed_2yr_total"] / (games_in_season * 2)).clip(0, 1)  # track record
        + 0.20 * df["soft_tissue_injury_flag"].astype(float)   # soft tissue = high recurrence
        + 0.15 * df["acl_history_flag"].astype(float)          # ACL history = cautionary
        + 0.05 * (df["concussion_history_count"].clip(0, 3) / 3)  # concussion accumulation
    )

    return df

In [45]:
join_df = _add_injury_features(join_df, injury_df)
join_df

,player_id,name,position,season,team,college_team,birth_date,status,height,weight,years_exp,draft_round,draft_pick,draft_team,rookie_year,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,carries,rushing_yards,rushing_tds,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_ppg_ppr,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game,snap_pct_std,snap_pct_mean,full_name,head_coach,offensive_coordinator,defensive_coordinator,offensive_scheme,plays_per_game,pass_rate,pass_rate_neutral,team_pass_yards,team_rush_yards,team_total_tds,team_pass_attempts,team_targets,points_per_game,offensive_line_rank,fantasy_ppg_last_season,fantasy_ppg_2yr_avg,fantasy_ppg_3yr_avg,fantasy_ppg_trend,career_games,targets_per_game,carries_per_game,snap_pct_trend,role_security_score,passing_epa_per_att,rushing_epa_per_carry,receiving_epa_per_tgt,yac_per_rec,cpoe,avg_time_to_throw,aggressiveness,avg_intended_air_yards,ryoe_per_att,efficiency,avg_time_to_los,avg_separation,avg_cushion,catch_pct_above_expectation,avg_yac_above_expectation,games_missed_last_season,games_missed_2yr_total,weighted_injury_score,soft_tissue_injury_flag,acl_history_flag,concussion_history_count,injury_risk_score
0,00-0007091,Matt Hasselbeck,QB,2015,IND,Boston College,1975-09-25,ACT,76,235,17,6.0,NaN,GB,1999,8,156,256,1690,9,5,-0.113061,0.609375,6.601562,83.951823,16,16,15,0,-5.870652,0.937500,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.10,11.387500,0.867500,0.000000,0.000000,NaN,0.000000,0.000000,0.197032,0.867500,Indianapolis Colts,Chuck Pagano,Rob Chudzinski,Greg Manusky,None,65.88,0.6243,0.5876,3928.0,1438.0,None,658.0,621.0,20.81,15.0,NaN,NaN,NaN,NaN,NaN,0.000000,2.000000,NaN,0.483750,-0.000442,-0.366916,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,1.3,False,False,0.0,0.020588
1,00-0010346,Peyton Manning,QB,2015,CAR,Tennessee,1976-03-24,ACT,77,230,18,1.0,NaN,IND,1998,10,198,331,2249,9,17,-31.956645,0.598187,6.794562,67.906596,16,6,-6,0,-3.616597,-1.000000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.36,9.136000,1.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.208431,0.914615,Carolina Panthers,Ron Rivera,Mike Shula,Sean McDermott,West Coast,66.63,0.5016,0.5259,4634.0,2696.0,None,635.0,593.0,31.25,22.0,NaN,NaN,NaN,NaN,NaN,0.000000,0.600000,NaN,0.515000,-0.096546,-0.602766,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,0.0,6.0,False,False,0.0,0.123529
2,00-0019596,Tom Brady,QB,2015,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,402,624,4770,36,7,127.536179,0.644231,7.644231,102.176816,38,34,53,3,6.768190,1.558824,1,1,36,0,3.129031,36.000000,1.000000,36.000000,7,29,344.70,21.543750,0.989444,0.017857,0.012259,5.142857,0.035367,0.062500,0.021549,0.989444,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,65.78,0.6503,0.6693,5424.0,1486.0,None,770.0,728.0,29.06,11.0,NaN,NaN,NaN,NaN,NaN,0.062500,2.125000,NaN,0.547847,0.204385,0.199064,3.129031,29.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.3,False,False,0.0,0.000000
3,00-0019596,Tom Brady,QB,2016,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,12,291,432,3554,28,2,147.296480,0.673611,8.226852,112.172068,15,28,64,0,-5.717147,2.285714,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,258.56,21.546667,0.972667,0.000000,0.000000,NaN,0.000000,0.000000,0.057379,0.972667,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.84,0.5640,0.5969,5593.0,2131.0,None,727.0,694.0,27.56,6.0,21.543750,21.543750,21.543750,NaN,16.0,0.000000,2.333333,-0.016778,0.544667,0.340964,-0.204184,NaN,NaN,3.269891,2.559618,17.592593,8.112685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.2,False,False,0.0,0.000000
4,00-0019596,Tom Brady,QB,2017,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,385,581,4577,32,8,138.508184,0.662651,7.877797,102

In [47]:
missing_pct = (
    join_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                        feature  missing_pct
0                team_total_tds     1.000000
1   catch_pct_above_expectation     1.000000
2                  ryoe_per_att     0.936639
3        avg_intended_air_yards     0.934670
4                aggressiveness     0.934670
5                          cpoe     0.934670
6             avg_time_to_throw     0.934670
7                    efficiency     0.918203
8               avg_time_to_los     0.918203
9                completion_pct     0.830857
10            yards_per_attempt     0.830857
11                passer_rating     0.830857
12          passing_epa_per_att     0.830857
13                  avg_cushion     0.797566
14               avg_separation     0.797566
15    avg_yac_above_expectation     0.797566
16            fantasy_ppg_trend     0.513514
17        rushing_epa_per_carry     0.431359
18              yards_per_carry     0.431359
19               snap_pct_trend     0.309289
20        weighted_injury_score     0.304815
21      fa

### Age Features

In [48]:
def _age_on_date(birth_date, as_of) -> Optional[float]:
    if pd.isna(birth_date):
        return None
    bd = birth_date.date() if hasattr(birth_date, "date") else birth_date
    return (as_of - bd).days / 365.25
    
def _add_age_features(df: pd.DataFrame, players_df: pd.DataFrame) -> pd.DataFrame:
        """
        Age relative to position peak is one of the most predictive features.
        A 28-year-old WR (2 years past peak) is a very different asset than
        a 23-year-old WR (3 years before peak) even with identical current stats.
        """
        players_df = players_df.copy()
        players_df["birth_date"] = pd.to_datetime(players_df["birth_date"], errors="coerce")

        if "birth_date" not in df.columns:
            df = df.merge(players_df[["player_id", "birth_date"]], on="player_id", how="left")

        df["birth_date"] = pd.to_datetime(df["birth_date"], errors="coerce")

        # Age as of September 1 of the given season (start of NFL season)
        df["age"] = df.apply(
            lambda r: _age_on_date(r["birth_date"], date(int(r["season"]), 9, 1))
            if pd.notna(r.get("birth_date")) else np.nan,
            axis=1,
        )

        # Age at NFL entry
        df["age_at_nfl_entry"] = df.apply(
            lambda r: _age_on_date(r["birth_date"], date(int(r["rookie_year"]), 9, 1))
            if pd.notna(r.get("birth_date")) and pd.notna(r.get("rookie_year")) else np.nan,
            axis=1,
        )

        # Age vs position peak (negative = still ascending, positive = declining)
        df["age_vs_position_peak"] = df.apply(
            lambda r: (r["age"] - POSITION_PEAK_AGE.get(r["position"], 27))
            if pd.notna(r.get("age")) else np.nan,
            axis=1,
        )

        return df

In [49]:
join_df = _add_age_features(join_df, players_df)
join_df

,player_id,name,position,season,team,college_team,birth_date,status,height,weight,years_exp,draft_round,draft_pick,draft_team,rookie_year,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,carries,rushing_yards,rushing_tds,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_ppg_ppr,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game,snap_pct_std,snap_pct_mean,full_name,head_coach,offensive_coordinator,defensive_coordinator,offensive_scheme,plays_per_game,pass_rate,pass_rate_neutral,team_pass_yards,team_rush_yards,team_total_tds,team_pass_attempts,team_targets,points_per_game,offensive_line_rank,fantasy_ppg_last_season,fantasy_ppg_2yr_avg,fantasy_ppg_3yr_avg,fantasy_ppg_trend,career_games,targets_per_game,carries_per_game,snap_pct_trend,role_security_score,passing_epa_per_att,rushing_epa_per_carry,receiving_epa_per_tgt,yac_per_rec,cpoe,avg_time_to_throw,aggressiveness,avg_intended_air_yards,ryoe_per_att,efficiency,avg_time_to_los,avg_separation,avg_cushion,catch_pct_above_expectation,avg_yac_above_expectation,games_missed_last_season,games_missed_2yr_total,weighted_injury_score,soft_tissue_injury_flag,acl_history_flag,concussion_history_count,injury_risk_score,age,age_at_nfl_entry,age_vs_position_peak
0,00-0007091,Matt Hasselbeck,QB,2015,IND,Boston College,1975-09-25,ACT,76,235,17,6.0,NaN,GB,1999,8,156,256,1690,9,5,-0.113061,0.609375,6.601562,83.951823,16,16,15,0,-5.870652,0.937500,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.10,11.387500,0.867500,0.000000,0.000000,NaN,0.000000,0.000000,0.197032,0.867500,Indianapolis Colts,Chuck Pagano,Rob Chudzinski,Greg Manusky,None,65.88,0.6243,0.5876,3928.0,1438.0,None,658.0,621.0,20.81,15.0,NaN,NaN,NaN,NaN,NaN,0.000000,2.000000,NaN,0.483750,-0.000442,-0.366916,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,1.3,False,False,0.0,0.020588,39.934292,23.934292,5.934292
1,00-0010346,Peyton Manning,QB,2015,CAR,Tennessee,1976-03-24,ACT,77,230,18,1.0,NaN,IND,1998,10,198,331,2249,9,17,-31.956645,0.598187,6.794562,67.906596,16,6,-6,0,-3.616597,-1.000000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.36,9.136000,1.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.208431,0.914615,Carolina Panthers,Ron Rivera,Mike Shula,Sean McDermott,West Coast,66.63,0.5016,0.5259,4634.0,2696.0,None,635.0,593.0,31.25,22.0,NaN,NaN,NaN,NaN,NaN,0.000000,0.600000,NaN,0.515000,-0.096546,-0.602766,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,0.0,6.0,False,False,0.0,0.123529,39.438741,22.439425,5.438741
2,00-0019596,Tom Brady,QB,2015,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,402,624,4770,36,7,127.536179,0.644231,7.644231,102.176816,38,34,53,3,6.768190,1.558824,1,1,36,0,3.129031,36.000000,1.000000,36.000000,7,29,344.70,21.543750,0.989444,0.017857,0.012259,5.142857,0.035367,0.062500,0.021549,0.989444,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,65.78,0.6503,0.6693,5424.0,1486.0,None,770.0,728.0,29.06,11.0,NaN,NaN,NaN,NaN,NaN,0.062500,2.125000,NaN,0.547847,0.204385,0.199064,3.129031,29.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.3,False,False,0.0,0.000000,38.078029,23.080082,4.078029
3,00-0019596,Tom Brady,QB,2016,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,12,291,432,3554,28,2,147.296480,0.673611,8.226852,112.172068,15,28,64,0,-5.717147,2.285714,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,258.56,21.546667,0.972667,0.000000,0.000000,NaN,0.000000,0.000000,0.057379,0.972667,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.84,0.5640,0.5969,5593.0,2131.0,None,727.0,694.0,27.56,6.0,21.543750,21.543750,21.543750,NaN,16.0,0.000000,2.333333,-0.016778,0.544667,0.340964,-0.204184,NaN,NaN,3.269891,2.559618,17.592593,8.112685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.2,False,False,0.0,0.000000,39.080082,2

In [51]:
missing_pct = (
    join_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                        feature  missing_pct
0                team_total_tds     1.000000
1   catch_pct_above_expectation     1.000000
2                  ryoe_per_att     0.936639
3             avg_time_to_throw     0.934670
4                          cpoe     0.934670
5                aggressiveness     0.934670
6        avg_intended_air_yards     0.934670
7                    efficiency     0.918203
8               avg_time_to_los     0.918203
9                 passer_rating     0.830857
10            yards_per_attempt     0.830857
11               completion_pct     0.830857
12          passing_epa_per_att     0.830857
13    avg_yac_above_expectation     0.797566
14                  avg_cushion     0.797566
15               avg_separation     0.797566
16            fantasy_ppg_trend     0.513514
17        rushing_epa_per_carry     0.431359
18              yards_per_carry     0.431359
19               snap_pct_trend     0.309289
20        weighted_injury_score     0.304815
21        

### Team Context Features

In [52]:
def _compute_scheme_fit(row) -> float:
    """
    Scheme fit: 0-1 score of how well player role matches team offensive system.
    High target share players fit best on high pass-rate teams.
    High carry players fit best on run-heavy teams.
    """
    pass_rate = row.get("pass_rate") or 0.5
    target_share = row.get("target_share") or 0
    carries_pg = row.get("carries_per_game") or 0
    position = row.get("position", "")

    if position in ("WR", "TE"):
        # WR/TE thrive on high pass-rate teams
        return min(1.0, target_share * 5 * pass_rate)
    elif position == "RB":
        # RBs thrive on run-heavy teams (1-pass_rate = run rate)
        run_rate = 1 - pass_rate
        return min(1.0, (carries_pg / 15) * (run_rate * 2))
    return 0.5

def _add_team_context_features(df: pd.DataFrame, team_df: pd.DataFrame) -> pd.DataFrame:
    """
    Team context can make or break a player's fantasy value.
    A WR on a run-heavy team has a lower ceiling regardless of skill.
    New OC or team change is a high variance signal.
    """
    # New team flag (compared to previous season)
    df = df.sort_values(["player_id", "season"])
    df["prev_team"] = df.groupby("player_id")["team"].shift(1)
    df["new_team_flag"] = (df["team"] != df["prev_team"]) & df["prev_team"].notna()

    # New OC flag — requires team_df to have OC info year-over-year
    if "offensive_coordinator" in df.columns:
        df["prev_oc"] = df.groupby("player_id")["offensive_coordinator"].shift(1)
        df["new_oc_flag"] = (
            (df["offensive_coordinator"] != df["prev_oc"]) & df["prev_oc"].notna()
        )
    else:
        df["new_oc_flag"] = False

    # Scheme fit score: how well does the player's style match the team's scheme?
    # Simplified version: high-target players should go to high pass-rate teams
    if "pass_rate" in df.columns:
        df["scheme_fit_score"] = df.apply(
            lambda r: _compute_scheme_fit(r), axis=1
        )
    else:
        df["scheme_fit_score"] = np.nan

    return df

In [53]:
join_df = _add_team_context_features(join_df, team_df)
join_df

,player_id,name,position,season,team,college_team,birth_date,status,height,weight,years_exp,draft_round,draft_pick,draft_team,rookie_year,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,carries,rushing_yards,rushing_tds,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_ppg_ppr,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game,snap_pct_std,snap_pct_mean,full_name,head_coach,offensive_coordinator,defensive_coordinator,offensive_scheme,plays_per_game,pass_rate,pass_rate_neutral,team_pass_yards,team_rush_yards,team_total_tds,team_pass_attempts,team_targets,points_per_game,offensive_line_rank,fantasy_ppg_last_season,fantasy_ppg_2yr_avg,fantasy_ppg_3yr_avg,fantasy_ppg_trend,career_games,targets_per_game,carries_per_game,snap_pct_trend,role_security_score,passing_epa_per_att,rushing_epa_per_carry,receiving_epa_per_tgt,yac_per_rec,cpoe,avg_time_to_throw,aggressiveness,avg_intended_air_yards,ryoe_per_att,efficiency,avg_time_to_los,avg_separation,avg_cushion,catch_pct_above_expectation,avg_yac_above_expectation,games_missed_last_season,games_missed_2yr_total,weighted_injury_score,soft_tissue_injury_flag,acl_history_flag,concussion_history_count,injury_risk_score,age,age_at_nfl_entry,age_vs_position_peak,prev_team,new_team_flag,prev_oc,new_oc_flag,scheme_fit_score
0,00-0007091,Matt Hasselbeck,QB,2015,IND,Boston College,1975-09-25,ACT,76,235,17,6.0,NaN,GB,1999,8,156,256,1690,9,5,-0.113061,0.609375,6.601562,83.951823,16,16,15,0,-5.870652,0.937500,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.10,11.387500,0.867500,0.000000,0.000000,NaN,0.000000,0.000000,0.197032,0.867500,Indianapolis Colts,Chuck Pagano,Rob Chudzinski,Greg Manusky,None,65.88,0.6243,0.5876,3928.0,1438.0,None,658.0,621.0,20.81,15.0,NaN,NaN,NaN,NaN,NaN,0.000000,2.000000,NaN,0.483750,-0.000442,-0.366916,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,1.3,False,False,0.0,0.020588,39.934292,23.934292,5.934292,NaN,False,NaN,False,0.500000
1,00-0010346,Peyton Manning,QB,2015,CAR,Tennessee,1976-03-24,ACT,77,230,18,1.0,NaN,IND,1998,10,198,331,2249,9,17,-31.956645,0.598187,6.794562,67.906596,16,6,-6,0,-3.616597,-1.000000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.36,9.136000,1.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.208431,0.914615,Carolina Panthers,Ron Rivera,Mike Shula,Sean McDermott,West Coast,66.63,0.5016,0.5259,4634.0,2696.0,None,635.0,593.0,31.25,22.0,NaN,NaN,NaN,NaN,NaN,0.000000,0.600000,NaN,0.515000,-0.096546,-0.602766,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,0.0,6.0,False,False,0.0,0.123529,39.438741,22.439425,5.438741,NaN,False,NaN,False,0.500000
2,00-0019596,Tom Brady,QB,2015,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,402,624,4770,36,7,127.536179,0.644231,7.644231,102.176816,38,34,53,3,6.768190,1.558824,1,1,36,0,3.129031,36.000000,1.000000,36.000000,7,29,344.70,21.543750,0.989444,0.017857,0.012259,5.142857,0.035367,0.062500,0.021549,0.989444,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,65.78,0.6503,0.6693,5424.0,1486.0,None,770.0,728.0,29.06,11.0,NaN,NaN,NaN,NaN,NaN,0.062500,2.125000,NaN,0.547847,0.204385,0.199064,3.129031,29.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.3,False,False,0.0,0.000000,38.078029,23.080082,4.078029,NaN,False,NaN,False,0.500000
3,00-0019596,Tom Brady,QB,2016,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,12,291,432,3554,28,2,147.296480,0.673611,8.226852,112.172068,15,28,64,0,-5.717147,2.285714,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,258.56,21.546667,0.972667,0.000000,0.000000,NaN,0.000000,0.000000,0.057379,0.972667,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.84,0.5640,0.5969,5593.0,2131.0,None,727.0,694.0,27.56,6.0,21.543750,21.543750,21.543750,NaN,16.0,0.000000,2.333333,-0.016778,0

In [54]:
missing_pct = (
    join_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                         feature  missing_pct
0    catch_pct_above_expectation     1.000000
1                 team_total_tds     1.000000
2                   ryoe_per_att     0.936639
3                           cpoe     0.934670
4         avg_intended_air_yards     0.934670
5                 aggressiveness     0.934670
6              avg_time_to_throw     0.934670
7                avg_time_to_los     0.918203
8                     efficiency     0.918203
9                 completion_pct     0.830857
10           passing_epa_per_att     0.830857
11                 passer_rating     0.830857
12             yards_per_attempt     0.830857
13                avg_separation     0.797566
14                   avg_cushion     0.797566
15     avg_yac_above_expectation     0.797566
16             fantasy_ppg_trend     0.513514
17               yards_per_carry     0.431359
18         rushing_epa_per_carry     0.431359
19                       prev_oc     0.346161
20                snap_pct_trend  

### Add Target Variable

In [55]:
def _add_target_variable(df: pd.DataFrame, stats_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add next season's fantasy PPG as the target variable.
    Uses a shift within each player's time series.
    This is what the ML model tries to predict.
    """
    next_season_ppg = (
        stats_df[["player_id", "season", "fantasy_ppg_ppr"]]
        .copy()
        .rename(columns={"season": "current_season", "fantasy_ppg_ppr": "fantasy_ppg_next_season"})
    )
    next_season_ppg["season"] = next_season_ppg["current_season"] - 1  # shift back 1 year
    df = df.merge(
        next_season_ppg[["player_id", "season", "fantasy_ppg_next_season"]],
        on=["player_id", "season"],
        how="left"
    )
    return df

In [56]:
join_df = _add_target_variable(join_df, stats_df)
join_df

,player_id,name,position,season,team,college_team,birth_date,status,height,weight,years_exp,draft_round,draft_pick,draft_team,rookie_year,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,carries,rushing_yards,rushing_tds,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_ppg_ppr,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game,snap_pct_std,snap_pct_mean,full_name,head_coach,offensive_coordinator,defensive_coordinator,offensive_scheme,plays_per_game,pass_rate,pass_rate_neutral,team_pass_yards,team_rush_yards,team_total_tds,team_pass_attempts,team_targets,points_per_game,offensive_line_rank,fantasy_ppg_last_season,fantasy_ppg_2yr_avg,fantasy_ppg_3yr_avg,fantasy_ppg_trend,career_games,targets_per_game,carries_per_game,snap_pct_trend,role_security_score,passing_epa_per_att,rushing_epa_per_carry,receiving_epa_per_tgt,yac_per_rec,cpoe,avg_time_to_throw,aggressiveness,avg_intended_air_yards,ryoe_per_att,efficiency,avg_time_to_los,avg_separation,avg_cushion,catch_pct_above_expectation,avg_yac_above_expectation,games_missed_last_season,games_missed_2yr_total,weighted_injury_score,soft_tissue_injury_flag,acl_history_flag,concussion_history_count,injury_risk_score,age,age_at_nfl_entry,age_vs_position_peak,prev_team,new_team_flag,prev_oc,new_oc_flag,scheme_fit_score,fantasy_ppg_next_season
0,00-0007091,Matt Hasselbeck,QB,2015,IND,Boston College,1975-09-25,ACT,76,235,17,6.0,NaN,GB,1999,8,156,256,1690,9,5,-0.113061,0.609375,6.601562,83.951823,16,16,15,0,-5.870652,0.937500,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.10,11.387500,0.867500,0.000000,0.000000,NaN,0.000000,0.000000,0.197032,0.867500,Indianapolis Colts,Chuck Pagano,Rob Chudzinski,Greg Manusky,None,65.88,0.6243,0.5876,3928.0,1438.0,None,658.0,621.0,20.81,15.0,NaN,NaN,NaN,NaN,NaN,0.000000,2.000000,NaN,0.483750,-0.000442,-0.366916,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,1.3,False,False,0.0,0.020588,39.934292,23.934292,5.934292,NaN,False,NaN,False,0.500000,NaN
1,00-0010346,Peyton Manning,QB,2015,CAR,Tennessee,1976-03-24,ACT,77,230,18,1.0,NaN,IND,1998,10,198,331,2249,9,17,-31.956645,0.598187,6.794562,67.906596,16,6,-6,0,-3.616597,-1.000000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.36,9.136000,1.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.208431,0.914615,Carolina Panthers,Ron Rivera,Mike Shula,Sean McDermott,West Coast,66.63,0.5016,0.5259,4634.0,2696.0,None,635.0,593.0,31.25,22.0,NaN,NaN,NaN,NaN,NaN,0.000000,0.600000,NaN,0.515000,-0.096546,-0.602766,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,0.0,6.0,False,False,0.0,0.123529,39.438741,22.439425,5.438741,NaN,False,NaN,False,0.500000,NaN
2,00-0019596,Tom Brady,QB,2015,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,402,624,4770,36,7,127.536179,0.644231,7.644231,102.176816,38,34,53,3,6.768190,1.558824,1,1,36,0,3.129031,36.000000,1.000000,36.000000,7,29,344.70,21.543750,0.989444,0.017857,0.012259,5.142857,0.035367,0.062500,0.021549,0.989444,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,65.78,0.6503,0.6693,5424.0,1486.0,None,770.0,728.0,29.06,11.0,NaN,NaN,NaN,NaN,NaN,0.062500,2.125000,NaN,0.547847,0.204385,0.199064,3.129031,29.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.3,False,False,0.0,0.000000,38.078029,23.080082,4.078029,NaN,False,NaN,False,0.500000,21.546667
3,00-0019596,Tom Brady,QB,2016,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,12,291,432,3554,28,2,147.296480,0.673611,8.226852,112.172068,15,28,64,0,-5.717147,2.285714,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,258.56,21.546667,0.972667,0.000000,0.000000,NaN,0.000000,0.000000,0.057379,0.972667,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,67.84,0.5640,0.5969,5593.0,2131.0,None,727.0,694.0,27.56,6.0,21.543750,21.543750,21.543

In [57]:
missing_pct = (
    join_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                         feature  missing_pct
0                 team_total_tds     1.000000
1    catch_pct_above_expectation     1.000000
2                   ryoe_per_att     0.936639
3                           cpoe     0.934670
4              avg_time_to_throw     0.934670
5                 aggressiveness     0.934670
6         avg_intended_air_yards     0.934670
7                avg_time_to_los     0.918203
8                     efficiency     0.918203
9            passing_epa_per_att     0.830857
10                 passer_rating     0.830857
11             yards_per_attempt     0.830857
12                completion_pct     0.830857
13                avg_separation     0.797566
14                   avg_cushion     0.797566
15     avg_yac_above_expectation     0.797566
16             fantasy_ppg_trend     0.513514
17         rushing_epa_per_carry     0.431359
18               yards_per_carry     0.431359
19                       prev_oc     0.346161
20       fantasy_ppg_next_season  

### College Stats Features

In [59]:
import unicodedata
import re
from rapidfuzz import process, fuzz
# from data.ingestion.constants import NICKNAME_MAP, SUFFIXES
# from utils.helpers import make_player_key 

NORMALIZED_COLLEGE_TEAMS = {
    "Lehigh University": "Lehigh",
    "Grambling State": "Grambling",
    "Mississippi": "Ole Miss",
    "Northern Michigan University": "Northern Michigan",
    "Jackson State University": "Jackson State",
    "Chadron State": "Chadron St",
    "Aurora University": "Aurora",
    "University of U. of Pacific": "Pacific",
    "University of Evansville": "Evansville",
    "Miami (Ohio)": "Miami (OH)",
    "Central Washington University": "Central Washington",
    "California State-Northridge": "Cal State Northridge",
    "Louisiana-Monroe": "UL Monroe",
    "St. Cloud State University": "St. Cloud State",
    "Fayetteville State University": "Fayetteville State",
    "University of Sonoma State": "Cal State Sonoma",
    "Cameron University": "Cameron",
    "Angelo State University": "Angelo State",
    "Central State University, Ohio": "Central State",
    "University of Arkansas at Pine Bluff": "Arkansas-Pine Bluff",
    "Nicholls State": "Nicholls",
    "Johnson C. Smith University": "Johnson C. Smith",
    "McNeese State": "McNeese",
    "University of St. Thomas": "St. Thomas (MN)",
    "Louisiana-Lafayette": "Lafayette",
    "N.C. State": "NC State",
    "Emporia State University": "Emporia State",
    "California State - Long Beach": "Cal State Long Beach",
    "Stephen F. Austin State": "Stephen F. Austin",
    "Middle Tennessee State": "Middle Tennessee",
    "Northwestern State (LA)": "Northwestern State",
    "New Mexico Highlands Univ.": "New Mexico Highlands",
    "Tennessee-Chattanooga": "Chattanooga",
    "Winston Salem State University": "Winston-Salem",
    "Alliant International University": "Alliant International",
    "University of New Haven": "New Haven",
    "Henderson State University": "Henderson State",
    "NW Missouri State University": "Northwest Missouri State",
    "Pomona State": "Pomona",
    "Southern Utah State": "Southern Utah",
    "Langston University": "Langston",
    "Mercyhurst College": "Mercyhurst",
    "Mississippi Valley State University": "Mississippi Valley State",
    "California Riverside, University of": "UC Riverside",
    "Santa Clara University": "Santa Clara",
    "North Greenville College": "North Greenville",
    "Indiana PA,  University of": "Indiana (PA)",
    "Cal Poly (San Luis Obispo)": "Cal Poly",
    "Penn": "Pennsylvania",
    "Delta State University": "Delta State",
    "California State-Fullerton": "Cal State Fullerton",
    "Towson State": "Towson",
    "Butler University": "Butler",
    "Western New Mexico University": "Western New Mexico",
    "Northeastern University, Mass": "Northeastern",
    "Bethune-Cookman College": "Bethune-Cookman",
    "Acadia University, Canada": "Acadia",
    "Taft College": "Taft",
    "Bradley University": "Bradley",
    "Lamar Community College": "Lamar CC",
    "Millikin University": "Millikin",
    "Missouri Valley College": "Missouri Valley",
    "Independence Comm. College": "Independence CC",
    "Rowan College": "Rowan",
    "Univ of Arkansas at Monticello": "Arkansas-Monticello",
    "Shasta College": "Shasta",
    "Walla Walla Community College": "Walla Walla CC",
    "Savannah State College": "Savannah State",
    "Morehead State University": "Morehead State",
    "San Joaquin Delta College": "San Joaquin Delta",
    "Wesleyan University": "Wesleyan",
    "Alvin Community College": "Alvin CC",
    "Northwestern Oklahoma State University": "Northwestern Oklahoma State",
    "Western Washington University": "Western Washington",
    "University of Millersville": "Millersville",
    "University of Cheyney": "Cheyney",
    "University of South Florida": "South Florida",
    "Sam Houston State": "Sam Houston",
    "Ohio Northern University": "Ohio Northern",
    "Middle Georgia College": "Middle Georgia",
    "Carroll College": "Carroll",
    "College of Eastern Utah": "Eastern Utah",
    "Northern State University": "Northern State",
    "Winona State University": "Winona State",
    "Tuskegee University": "Tuskegee",
    "Black Hills State University": "Black Hills State",
    "Eastern Oregon State Univ.": "Eastern Oregon State",
    "Charleston Southern University": "Charleston Southern",
    "Utah St.": "Utah State",
    "Miami (FL)": "Miami",
    "Fresno State": "Fresno State",
    "Michigan State": "Michigan State",
    "Kansas State": "Kansas State",
    "San Diego State": "San Diego State",
    "Johns Hopkins University": "Johns Hopkins",
    "Mississippi St.": "Mississippi State",
    "Ohio St.": "Ohio State",
    "Boise St.": "Boise State",
    "Monmouth College": "Monmouth",
    "Appalachian St.": "Appalachian State",
    "North Carolina St.": "North Carolina State",
    "Oklahoma St.": "Oklahoma State",
    "Arizona St.": "Arizona State",
    "Bemidji State University": "Bemidji State",
    "Washington St.": "Washington State",
    "Jacksonville St.": "Jacksonville State",
    "Grand Valley St.": "Grand Valley State",
    "Florida St.": "Florida State",
    "Oregon St.": "Oregon State",
    "Colorado St.": "Colorado State",
    "Ball St.": "Ball State",
    "Saginaw Valley St.": "Saginaw Valley State",
    "Pittsburg St.": "Pittsburg State",
    "Boston Col.": "Boston College",
    "San Jose St.": "San Jose State",
    "Murray St.": "Murray State",
    "Kent St.": "Kent State",
    "Penn St.": "Penn State",
    "Univ of California-San Diego": "UC San Diego",
    "Ala-Birmingham": "Alabama-Birmingham",
    "Illinois St.": "Illinois State",
    "University of Rochester": "Rochester",
    "Montana St.": "Montana State",
    "North Dakota St.": "North Dakota State",
    "South Carolina St.": "South Carolina State",
    "Virginia Commonwealth Univ.": "Virginia Commonwealth",
    "Georgia St.": "Georgia State",
    "Grambling St.": "Grambling",
    "Barry University": "Barry",
    "Virginia St.": "Virginia State",
    "New Mexico St.": "New Mexico State",
    "Middle Tenn. St.": "Middle Tennessee State",
    "South Dakota St.": "South Dakota State",
    "La-Monroe": "UL Monroe",
    "Iowa St.": "Iowa State",
    "LSU (Shreveport)": "LSU Shreveport",
    "Loyola University": "Loyola",
    "Central Missouri St.": "Central Missouri State",
    "Youngstown St.": "Youngstown State",
    "University of West Florida": "West Florida",
    "Oklahoma Baptist University": "Oklahoma Baptist",
    "North Carolina State": "NC State",
    "Truman State University": "Truman State",
    "Shenandoah University": "Shenandoah",
    "UMass, Amherst": "UMass",
    "Louisiana State": "LSU",
    "University at Albany": "Albany",
    "Monmouth, N.J.": "Monmouth",
    "Southern California": "USC",
    "Southern Cal": "USC",
    "Miami (Fla.)": "Miami",
    "Nevada-Las Vegas": "UNLV",
    "St. John's University": "St. John's",
    "Montclair State College": "Montclair State",
    "Valparaiso University": "Valparaiso",
    "Michigan St.": "Michigan State",
}

NICKNAME_MAP = {
    "rob":        "robert",
    "bob":        "robert",
    "bobby":      "robert",
    "bill":       "william",
    "billy":      "william",
    "chris":      "christopher",
    "don":        "donald",
    "will":       "william",
    "willie":     "william",
    "mike":       "michael",
    "micky":      "michael",
    "mick":       "michael",
    "chris":      "christopher",
    "tj":         "tj",          # keep initials as-is
    "cj":         "cj",
    "dj":         "dj",
    "ab":         "ab",
    "aj":         "aj",
    "jj":         "jj",
    "jr":         "jr",
    "sr":         "sr",
    "jim":        "james",
    "jimmy":      "james",
    "jamie":      "james",
    "joe":        "joseph",
    "joey":       "joseph",
    "dan":        "daniel",
    "danny":      "daniel",
    "dave":       "david",
    "davey":      "david",
    "tom":        "thomas",
    "tommy":      "thomas",
    "tim":        "timothy",
    "timmy":      "timothy",
    "sam":        "samuel",
    "sammy":      "samuel",
    "ben":        "benjamin",
    "benny":      "benjamin",
    "nick":       "nicholas",
    "nicky":      "nicholas",
    "rick":       "richard",
    "ricky":      "richard",
    "rich":       "richard",
    "dick":       "richard",
    "alex":       "alexander",
    "zach":       "zachary",
    "zack":       "zachary",
    "matt":       "matthew",
    "matty":      "matthew",
    "jeff":       "jeffrey",
    "pat":        "patrick",
    "tony":       "anthony",
    "ken":        "kenneth",
    "kenny":      "kenneth",
    "ed":         "edward",
    "eddie":      "edward",
    "ted":        "edward",
    "tommie":     "thomas",
    "tommy":      "thomas",
    "tom":        "thomas",
    "fred":       "frederick",
    "freddie":    "frederick",
    "greg":       "gregory",
    "larry":      "lawrence",
    "steve":      "steven",
    "stevie":     "steven",
    "charlie":    "charles",
    "chuck":      "charles",
    "dak":        "dak",         # keep unusual names as-is
    "odell":      "odell",
    "ty":         "ty",
    "gabe":       "gabriel",
}

# Suffixes to strip from names (they don't appear consistently across sources)
SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "v"}

def _unicode_to_ascii(text: str) -> str:
    """Convert accented / special unicode chars to closest ASCII equivalent."""
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
 
 
def make_player_key(name: str) -> str:
    """
    Convert a raw player name string into a normalized, deterministic key
    suitable for joining across CFBD and NFL datasets.
 
    Steps:
        1. Unicode → ASCII  (handles é, ñ, etc.)
        2. Lower-case
        3. Strip punctuation (apostrophes, dots, dashes, etc.)
        4. Remove suffixes (Jr., Sr., II, III …)
        5. Expand known nicknames / abbreviations (Bill → william)
        6. Rejoin and return a single lowercase string
 
    Examples:
        "D.J. Moore"     →  "dj moore"
        "De'Von Achane"  →  "devon achane"
        "Odell Beckham Jr." → "odell beckham"
        "Patrick Mahomes II" → "patrick mahomes"
        "Will Levis"     →  "william levis"
    """
    if not isinstance(name, str) or not name.strip():
        return ""
 
    # 1. ASCII transliteration
    name = _unicode_to_ascii(name)
 
    # 2. Lower-case
    name = name.lower()
 
    # 3. Remove punctuation: keep only letters, digits, and spaces.
    #    This collapses "D.J." → "dj", "O'Dell" → "odell", "Ja'Marr" → "jamarrr"
    name = re.sub(r"[^a-z0-9\s]", "", name)
 
    # 4. Collapse extra whitespace
    tokens = name.split()
 
    # 5. Remove suffix tokens
    tokens = [t for t in tokens if t not in SUFFIXES]
 
    # 6. Expand nicknames (first name only, i.e. tokens[0])
    if tokens:
        tokens[0] = NICKNAME_MAP.get(tokens[0], tokens[0])
 
    return "_".join(tokens)

import math

def _add_college_features(
        df: pd.DataFrame,
        college_df: pd.DataFrame,
        combine_df: pd.DataFrame,
        players_df: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        For incoming rookies and recent draft picks, add college production
        and athleticism features. These are NULL for veterans.

        Key metrics:
          - Dominator Rating: player's share of team receiving production (>30% = elite)
          - Breakout Age: age at first 20%+ dominator season (younger = better)
          - Competition adjustment: P5 stats weighted higher than FCS
          - Athleticism composites from combine
        """
        if college_df.empty:
            return df

        college_df = college_df.copy()

        # ---- Dominator Rating ----
        # = (player_rec_yards / team_rec_yards + player_rec_tds / team_rec_tds) / 2
        # Capped at 100% to handle edge cases
        college_df["dominator_rating"] = (
            (college_df["rec_yards"] / college_df["team_rec_yards"].replace(0, np.nan)).clip(0, 1)
            + (college_df["rec_tds"] / college_df["team_rec_tds"].replace(0, np.nan)).clip(0, 1)
        ) / 2

        # Competition adjustment
        college_df["conference_tier"] = college_df["conference"].apply(_classify_conference)
        college_df["adj_multiplier"] = college_df["conference_tier"].map(CONFERENCE_MULTIPLIER)
        college_df["adjusted_dominator"] = college_df["dominator_rating"] * college_df["adj_multiplier"]

        # ---- Per-game production ----
        def _calc_yards_per_game(row):
            games_played = row["games_played"]
            games_played = np.nan if games_played == 0 else games_played
            if row["position"] == "QB":
                return row["pass_yards"] / games_played
            elif row["position"] == "RB":
                return row["rush_yards"] / games_played
            else:
                return row["rec_yards"] / games_played

        def _calc_tds_per_game(row):
            games_played = row["games_played"]
            games_played = np.nan if games_played == 0 else games_played
            if row["position"] == "QB":
                return row["pass_tds"] / games_played
            elif row["position"] == "RB":
                return row["rush_tds"] / games_played
            else:
                return row["rec_tds"] / games_played

        college_df["college_yards_per_game"] = college_df.apply(_calc_yards_per_game, axis=1)
        college_df["college_tds_per_game"] = college_df.apply(_calc_tds_per_game, axis=1)

        # ---- Breakout Age ----
        # Earliest season a player hit 20%+ dominator rating
        # Requires knowing player age at that season — join with player birth dates
        players_slim = (
            players_df[["player_id", "name", "college_team", "position", "birth_date"]]
            .copy()
            .rename(columns={"college_team": "team", "name":"player_name"})
            .groupby(["player_name", "position", "team"]).first().reset_index()
        )
        players_slim["birth_date"] = pd.to_datetime(players_slim["birth_date"], errors="coerce")
        players_slim['team'] = players_slim['team'].str.split('; ')
        players_slim = players_slim.explode("team")
        players_slim["team"] = players_slim["team"].apply(lambda x: NORMALIZED_COLLEGE_TEAMS.get(x, x))
        players_slim["norm_player_key"] = players_slim["player_name"].apply(make_player_key) + "_" + players_slim["position"].str.lower() + "_" + players_slim["team"].apply(make_player_key)
        players_slim = players_slim.sort_values("player_id", ascending=True).groupby("norm_player_key").first().reset_index()

        # Normalize cfbd id dtypes to avoid int64 vs object merge errors
        if "cfbd_player_id" in college_df.columns:
            college_df["cfbd_player_id"] = college_df["cfbd_player_id"].astype("string")
        else:
            college_df["cfbd_player_id"] = pd.Series(dtype="string")

        college_df["norm_player_key"] = college_df["player_name"].apply(make_player_key) + "_" + college_df["position"].str.lower() + "_" + college_df["team"].apply(make_player_key)
        college_with_player = college_df.drop("player_id", axis=1).merge(players_slim[["norm_player_key", "player_id", "birth_date"]], on=["norm_player_key"], how="inner")
        college_with_player["age_at_season"] = college_with_player.apply(
            lambda r: _age_on_date(r["birth_date"], date(int(r["season"]), 9, 1))
            if pd.notna(r.get("birth_date")) else np.nan,
            axis=1,
        )
        breakout = (
            college_with_player[college_with_player["dominator_rating"] >= 0.20]
            .groupby("player_id")["age_at_season"]
            .min()
            .reset_index(name="breakout_age")
        )

        # Best college season (highest adjusted dominator)
        best_college_season = (
            college_with_player
            .sort_values("adjusted_dominator", ascending=False)
            .groupby("player_id")
            .first()
            .reset_index()
            [["player_id", "cfbd_player_id", "adjusted_dominator", "college_yards_per_game",
              "college_tds_per_game", "conference_tier"]]
            .rename(columns={
                "adjusted_dominator": "dominator_rating",
                "conference_tier": "college_conference_tier"
            })
        )

        # ---- Combine / athleticism ----
        combine_slim = combine_df[[
            "player_id", "sparq_score", "relative_athletic_score",
            "speed_score", "forty_yard", "vertical_jump", "bmi"
        ]].copy()

        # Merge college features into main df via player_id + rookie flag
        # rookies are identified by years_exp == 0 or 1
        df["is_rookie_or_sophomore"] = df["years_exp"].isin([0, 1])

        best_college_season = best_college_season.merge(breakout, on="player_id", how="left")
        best_college_season = best_college_season.merge(combine_slim, on="player_id", how="left")    
        df = df.merge(
            best_college_season[[
                "player_id", "dominator_rating", "college_yards_per_game",
                "college_tds_per_game", "college_conference_tier", "breakout_age",
                "sparq_score", "relative_athletic_score", "speed_score",
                "forty_yard", "vertical_jump", "bmi"
            ]],
            on="player_id",
            how="left"
        )

        # Normalize draft pick within round
        df["draft_pick_normalized"] = df["draft_pick"].apply(
            lambda p: _normalize_draft_pick(p) if pd.notna(p) else np.nan
        )

        return df

def _classify_conference(conference: Optional[str]) -> int:
    """Return conference tier (1=P5, 2=G5, 3=FCS/other)."""
    if not conference:
        return 2
    p5 = {"SEC", "Big Ten", "Big 12", "ACC", "Pac-12", "Pac-10"}
    g5 = {"American Athletic", "Mountain West", "Conference USA", "MAC", "Sun Belt"}
    if conference in p5:
        return 1
    elif conference in g5:
        return 2
    return 3

def _normalize_draft_pick(pick: float) -> float:
    if not pick or pick <= 0:
        return 0.0
    return max(0.0, 1.0 - (math.log(float(pick)) / math.log(300)))

In [60]:
college_with_player = _add_college_features(join_df, college_df, combine_df, players_df)
college_with_player

,player_id,name,position,season,team,college_team,birth_date,status,height,weight,years_exp,draft_round,draft_pick,draft_team,rookie_year,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,carries,rushing_yards,rushing_tds,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_ppg_ppr,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game,snap_pct_std,snap_pct_mean,full_name,head_coach,offensive_coordinator,defensive_coordinator,offensive_scheme,plays_per_game,pass_rate,pass_rate_neutral,team_pass_yards,team_rush_yards,team_total_tds,team_pass_attempts,team_targets,points_per_game,offensive_line_rank,fantasy_ppg_last_season,fantasy_ppg_2yr_avg,fantasy_ppg_3yr_avg,fantasy_ppg_trend,career_games,targets_per_game,carries_per_game,snap_pct_trend,role_security_score,passing_epa_per_att,rushing_epa_per_carry,receiving_epa_per_tgt,yac_per_rec,cpoe,avg_time_to_throw,aggressiveness,avg_intended_air_yards,ryoe_per_att,efficiency,avg_time_to_los,avg_separation,avg_cushion,catch_pct_above_expectation,avg_yac_above_expectation,games_missed_last_season,games_missed_2yr_total,weighted_injury_score,soft_tissue_injury_flag,acl_history_flag,concussion_history_count,injury_risk_score,age,age_at_nfl_entry,age_vs_position_peak,prev_team,new_team_flag,prev_oc,new_oc_flag,scheme_fit_score,fantasy_ppg_next_season,is_rookie_or_sophomore,dominator_rating,college_yards_per_game,college_tds_per_game,college_conference_tier,breakout_age,sparq_score,relative_athletic_score,speed_score,forty_yard,vertical_jump,bmi,draft_pick_normalized
0,00-0007091,Matt Hasselbeck,QB,2015,IND,Boston College,1975-09-25,ACT,76,235,17,6.0,NaN,GB,1999,8,156,256,1690,9,5,-0.113061,0.609375,6.601562,83.951823,16,16,15,0,-5.870652,0.937500,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.10,11.387500,0.867500,0.000000,0.000000,NaN,0.000000,0.000000,0.197032,0.867500,Indianapolis Colts,Chuck Pagano,Rob Chudzinski,Greg Manusky,None,65.88,0.6243,0.5876,3928.0,1438.0,None,658.0,621.0,20.81,15.0,NaN,NaN,NaN,NaN,NaN,0.000000,2.000000,NaN,0.483750,-0.000442,-0.366916,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,1.3,False,False,0.0,0.020588,39.934292,23.934292,5.934292,NaN,False,NaN,False,0.500000,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00-0010346,Peyton Manning,QB,2015,CAR,Tennessee,1976-03-24,ACT,77,230,18,1.0,NaN,IND,1998,10,198,331,2249,9,17,-31.956645,0.598187,6.794562,67.906596,16,6,-6,0,-3.616597,-1.000000,0,0,0,0,0.000000,NaN,NaN,NaN,0,0,91.36,9.136000,1.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.208431,0.914615,Carolina Panthers,Ron Rivera,Mike Shula,Sean McDermott,West Coast,66.63,0.5016,0.5259,4634.0,2696.0,None,635.0,593.0,31.25,22.0,NaN,NaN,NaN,NaN,NaN,0.000000,0.600000,NaN,0.515000,-0.096546,-0.602766,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,0.0,6.0,False,False,0.0,0.123529,39.438741,22.439425,5.438741,NaN,False,NaN,False,0.500000,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00-0019596,Tom Brady,QB,2015,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.0,NWE,2000,16,402,624,4770,36,7,127.536179,0.644231,7.644231,102.176816,38,34,53,3,6.768190,1.558824,1,1,36,0,3.129031,36.000000,1.000000,36.000000,7,29,344.70,21.543750,0.989444,0.017857,0.012259,5.142857,0.035367,0.062500,0.021549,0.989444,New England Patriots,Bill Belichick,Josh McDaniels,Matt Patricia,Erhardt-Perkins,65.78,0.6503,0.6693,5424.0,1486.0,None,770.0,728.0,29.06,11.0,NaN,NaN,NaN,NaN,NaN,0.062500,2.125000,NaN,0.547847,0.204385,0.199064,3.129031,29.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.3,False,False,0.0,0.000000,38.078029,23.080082,4.078029,NaN,False,NaN,False,0.500000,21.546667,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.071966
3,00-0019596,Tom Brady,QB,2016,NE,Michigan,1977-08-03,ACT,76,225,23,6.0,199.

## Loading in Engineered Features
___
This is the table generated through the pipeline

In [75]:
def load_engineered_features(seasons):
    query = f"""
    select *
    from engineered_features
    WHERE season in ({','.join(map(str, seasons + [s-1 for s in seasons]))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

In [76]:
eng_feat_df = load_engineered_features(seasons)
eng_feat_df

,id,player_id,season,position,player_type,fantasy_ppg_ppr,fantasy_ppg_last_season,fantasy_ppg_2yr_avg,fantasy_ppg_3yr_avg,fantasy_ppg_trend,games_played,career_games,target_share,air_yards_share,wopr,snap_pct,snap_pct_trend,carries_per_game,targets_per_game,yards_per_target,yards_per_carry,yards_after_catch_per_rec,racr,epa_per_play,cpoe,ryoe_per_att,separation_avg,catch_pct_above_expected,games_missed_last_season,games_missed_2yr_total,injury_risk_score,soft_tissue_injury_flag,acl_history_flag,concussion_history_count,injury_designation_count,age,age_at_nfl_entry,years_experience,age_vs_position_peak,team_pass_rate,team_pass_rate_neutral,team_plays_per_game,team_pass_attempts,team_points_per_game,offensive_line_rank,new_team_flag,new_oc_flag,scheme_fit_score,dominator_rating,breakout_age,college_yards_per_game,college_tds_per_game,college_conference_tier,draft_round,draft_pick_normalized,sparq_score,relative_athletic_score,speed_score,height_weight_bmi,forty_yard,vertical_jump,fantasy_ppg_next_season
0,1,00-0007091,2015,QB,nfl,11.387500,NaN,NaN,NaN,NaN,8.0,NaN,0.000000,0.000000,0.000000,0.867500,NaN,2.000000,0.000000,NaN,0.937500,NaN,NaN,NaN,NaN,NaN,NaN,None,1,0,0.020588,0,0,0,2,39.934292,23.934292,17,5.934292,0.6243,0.5876,65.88,658.0,20.81,15.0,0,0,0.500000,NaN,NaN,NaN,NaN,NaN,6.0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
1,2,00-0010346,2015,QB,nfl,9.136000,NaN,NaN,NaN,NaN,10.0,NaN,0.000000,0.000000,0.000000,1.000000,NaN,0.600000,0.000000,NaN,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,None,6,0,0.123529,0,0,0,6,39.438741,22.439425,18,5.438741,0.5016,0.5259,66.63,635.0,31.25,22.0,0,0,0.500000,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
2,3,00-0019596,2015,QB,nfl,21.543750,NaN,NaN,NaN,NaN,16.0,NaN,0.017857,0.012259,0.035367,0.989444,NaN,2.125000,0.062500,36.000000,1.558824,29.000000,5.142857,3.129031,NaN,NaN,NaN,None,0,0,0.000000,0,0,0,1,38.078029,23.080082,23,4.078029,0.6503,0.6693,65.78,770.0,29.06,11.0,0,0,0.500000,NaN,NaN,NaN,NaN,NaN,6.0,0.071966,NaN,NaN,NaN,None,NaN,NaN,21.546667
3,4,00-0019596,2016,QB,nfl,21.546667,21.543750,21.543750,21.543750,NaN,12.0,16.0,0.000000,0.000000,0.000000,0.972667,-0.016778,2.333333,0.000000,NaN,2.285714,NaN,NaN,NaN,3.269891,NaN,NaN,None,0,0,0.000000,0,0,0,4,39.080082,23.080082,23,5.080082,0.5640,0.5969,67.84,727.0,27.56,6.0,0,0,0.500000,NaN,NaN,NaN,NaN,NaN,6.0,0.071966,NaN,NaN,NaN,None,NaN,NaN,18.492500
4,5,00-0019596,2017,QB,nfl,18.492500,21.546667,21.545208,21.545208,0.002917,16.0,28.0,0.000000,0.000000,0.000000,0.980526,0.007860,1.562500,0.000000,NaN,1.120000,NaN,NaN,NaN,0.849809,NaN,NaN,None,0,0,0.000000,0,0,0,1,40.079398,23.080082,23,6.079398,0.5984,0.6057,67.63,769.0,28.62,9.0,0,0,0.500000,NaN,NaN,NaN,NaN,NaN,6.0,0.071966,NaN,NaN,NaN,None,NaN,NaN,17.581250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5582,5583,00-0039917,2024,QB,nfl,8.820000,NaN,NaN,NaN,NaN,5.0,NaN,0.000000,0.000000,0.000000,0.664000,NaN,1.400000,0.000000,NaN,1.571429,NaN,NaN,NaN,NaN,NaN,NaN,None,0,0,0.000000,0,0,0,0,24.317591,24.317591,3,-9.682409,0.5440,0.5314,64.12,593.0,22.88,6.0,0,0,0.500000,0.000000,NaN,326.866667,2.400000,1.0,1.0,0.635428,NaN,7.932432,NaN,None,NaN,NaN,NaN
5583,5584,00-0039918,2024,QB,nfl,14.972941,NaN,NaN,NaN,NaN,17.0,NaN,0.000000,0.000000,0.000000,0.987647,NaN,4.764706,0.000000,NaN,6.037037,NaN,NaN,NaN,-1.126352,NaN,NaN,None,0,0,0.000000,0,0,0,0,22.787132,22.787132,3,-11.212868,0.5991,0.5340,63.24,644.0,18.24,32.0,0,0,0.500000,0.001733,NaN,302.750000,2.500000,1.0,1.0,1.000000,NaN,8.020548,NaN,None,NaN,NaN,NaN
5584,5585,00-0039919,2024,WR,nfl,8.523529,NaN,NaN,NaN,NaN,17.0,NaN,3.215179,5.542503,8.702520,0.835882,NaN,0.176471,5.941176,7.267327,5.000000,4.685185,0.525036,0.159021,NaN,NaN,2.960735,None,0,0,0.000000,0,0,0,1,22.247775,22.247775,3,-3.752225,0.5991,0.5340,63.24,644.0,18.24,32.0,0,0,1.000

In [77]:
eng_feat_df["injury_designation_count"]

0       2
1       6
2       1
3       4
4       1
       ..
5582    0
5583    0
5584    1
5585    0
5586    3
Name: injury_designation_count, Length: 5587, dtype: int64

In [10]:
players_df[players_df["player_id"] == "00-0039921"]

,player_id,sleeper_id,gsis_id,cfbd_id,name,first_name,last_name,position,nfl_team,college_team,status,birth_date,height,weight,years_exp,rookie_year,draft_round,draft_pick,draft_team,draft_year,is_rookie,depth_chart_order,jersey_number,updated_at
539,00-0039921,None,00-0039921,None,Trey Benson,Trey,Benson,RB,None,Florida St.,ACT,2002-07-23,72.0,220.0,3,2024,3.0,66.0,ARI,2024.0,0,None,33.0,2026-05-18 18:01:50.929926


In [80]:
players_df

,player_id,sleeper_id,gsis_id,cfbd_id,name,first_name,last_name,position,nfl_team,college_team,status,birth_date,height,weight,years_exp,rookie_year,draft_round,draft_pick,draft_team,draft_year,is_rookie,depth_chart_order,jersey_number,updated_at
0,00-0038389,None,00-0038389,None,Israel Abanikanda,Israel,Abanikanda,RB,None,Pittsburgh,ACT,2002-10-05,70.0,216.0,3,2023,5.0,143.0,NYJ,2023.0,0,None,30.0,2026-05-18 18:01:50.872218
1,00-0031021,None,00-0031021,None,Jared Abbrederis,Jared,Abbrederis,WR,None,Wisconsin,CUT,1990-12-17,73.0,195.0,4,2014,5.0,176.0,GNB,2014.0,0,None,10.0,2026-05-18 18:01:50.883150
2,00-0032104,None,00-0032104,None,Ameer Abdullah,Ameer,Abdullah,RB,None,Nebraska,UFA,1993-06-13,69.0,203.0,12,2015,2.0,54.0,DET,2015.0,0,None,26.0,2026-05-18 18:01:50.883274
3,00-0000007,None,00-0000007,None,Rabih Abdullah,Rabih,Abdullah,RB,None,Lehigh University,ACT,1975-04-27,72.0,235.0,7,1998,NaN,NaN,None,NaN,0,None,27.0,2026-05-18 18:01:50.883384
4,ABE498348,None,ABE498348,None,Walter Abercrombie,Walter,Abercrombie,RB,None,Baylor,ACT,1959-09-26,72.0,207.0,7,1982,1.0,NaN,PIT,NaN,0,None,0.0,2026-05-18 18:01:50.883487
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8337,00-0036100,None,00-0036100,None,Isaiah Zuber,Willie,Zuber,WR,None,Mississippi State; Kansas State,DEV,1997-04-15,72.0,190.0,1,2020,NaN,NaN,None,NaN,0,None,89.0,2026-05-18 18:01:51.606749
8338,00-0034052,None,00-0034052,None,Brandon Zylstra,Brandon,Zylstra,WR,None,Concordia College; Augustana (SD),DEV,1993-03-25,74.0,215.0,5,2018,NaN,NaN,None,NaN,0,None,13.0,2026-05-18 18:01:51.606831
8339,00-0036534,None,00-0036534,None,Shane Zylstra,Shane,Zylstra,TE,None,Minnesota State,UFA,1996-11-16,76.0,244.0,5,2021,NaN,NaN,None,NaN,0,None,84.0,2026-05-18 18:01:51.606913
8340,00-0000003,None,00-0000003,None,Abdul-Karim al-Jabbar,Abdul-Karim,al-Jabbar,RB,None,UCLA,CUT,1974-06-28,71.0,205.0,5,1996,NaN,NaN,None,NaN,0,None,34.0,2026-05-18 18:01:51.606995


In [542]:
eng_feat_df.groupby("player_id").size().sort_values(ascending=False)

player_id
00-0030564    10
00-0031345    10
00-0032268    10
00-0029604    10
00-0024243    10
              ..
00-0034169     1
00-0034192     1
00-0034207     1
00-0034221     1
00-0039921     1
Length: 1669, dtype: int64

In [543]:
eng_feat_df[eng_feat_df["player_id"] == "00-0030564"]

,id,player_id,season,position,player_type,fantasy_ppg_ppr,fantasy_ppg_last_season,fantasy_ppg_2yr_avg,fantasy_ppg_3yr_avg,fantasy_ppg_trend,games_played,career_games,target_share,air_yards_share,wopr,snap_pct,snap_pct_trend,carries_per_game,targets_per_game,yards_per_target,yards_per_carry,yards_after_catch_per_rec,racr,epa_per_play,cpoe,ryoe_per_att,separation_avg,catch_pct_above_expected,games_missed_last_season,games_missed_2yr_total,injury_risk_score,soft_tissue_injury_flag,acl_history_flag,concussion_history_count,injury_designation_count,age,age_at_nfl_entry,years_experience,age_vs_position_peak,team_pass_rate,team_pass_rate_neutral,team_plays_per_game,team_pass_attempts,team_points_per_game,offensive_line_rank,new_team_flag,new_oc_flag,scheme_fit_score,dominator_rating,breakout_age,college_yards_per_game,college_tds_per_game,college_conference_tier,draft_round,draft_pick_normalized,sparq_score,relative_athletic_score,speed_score,height_weight_bmi,forty_yard,vertical_jump,fantasy_ppg_next_season
1356,1357,00-0030564,2015,WR,nfl,20.693750,NaN,6.662500,8.193750,NaN,16.0,NaN,5.112128,7.518476,12.931126,0.967059,NaN,0.000000,12.000000,7.921875,NaN,2.027027,0.560427,0.327740,NaN,NaN,NaN,None,0,0,0.000000,0,0,0,None,23.236140,21.237509,14,-2.763860,0.5822,0.5511,70.12,694.0,21.19,14.0,0,0,1.0,0.393022,20.238193,NaN,NaN,1.0,1.0,0.422166,70.648121,9.239443,98.124842,None,4.57,36.0,12.337500
1357,1358,00-0030564,2016,WR,nfl,12.337500,20.693750,20.693750,13.678125,NaN,16.0,16.0,4.187702,5.744720,10.302858,0.969444,0.002386,0.000000,9.437500,6.317881,NaN,3.576923,0.544521,-0.014235,NaN,NaN,2.256239,None,0,0,0.000000,0,0,0,None,24.238193,21.237509,14,-1.761807,0.5663,0.5159,67.00,683.0,17.44,10.0,0,0,1.0,0.393022,20.238193,NaN,NaN,1.0,1.0,0.422166,70.648121,9.239443,98.124842,None,4.57,36.0,20.653333
1358,1359,00-0030564,2017,WR,nfl,20.653333,12.337500,16.515625,16.515625,-8.356250,15.0,32.0,5.362399,6.987684,12.934977,0.982000,0.012556,0.000000,11.600000,7.919540,NaN,3.739583,0.617107,0.217823,NaN,NaN,2.088425,None,1,0,0.020588,0,0,0,None,25.237509,21.237509,14,-0.762491,0.5641,0.5503,64.38,581.0,21.12,31.0,0,1,1.0,0.393022,20.238193,NaN,NaN,1.0,1.0,0.422166,70.648121,9.239443,98.124842,None,4.57,36.0,20.843750
1359,1360,00-0030564,2018,WR,nfl,20.843750,20.653333,16.495417,17.894861,-0.020208,16.0,47.0,5.347734,7.320739,13.146118,0.990588,0.008588,0.062500,10.187500,9.644172,-7.0,3.365217,0.841091,0.467406,NaN,NaN,2.522421,None,0,0,0.000000,0,0,0,None,26.236824,21.237509,14,0.236824,0.5600,0.5411,65.24,621.0,25.12,32.0,0,0,1.0,0.393022,20.238193,NaN,NaN,1.0,1.0,0.422166,70.648121,9.239443,98.124842,None,4.57,36.0,17.902667
1360,1361,00-0030564,2019,WR,nfl,17.902667,20.843750,20.748542,17.944861,4.253125,15.0,63.0,4.678010,5.462771,10.840955,0.968824,-0.021765,0.133333,10.000000,7.766667,9.0,3.721154,0.767963,0.392374,NaN,NaN,2.672764,None,0,0,0.000000,0,0,0,None,27.236140,21.237509,14,1.236140,0.5795,0.5441,64.61,674.0,23.62,30.0,0,0,1.0,0.393022,20.238193,NaN,NaN,1.0,1.0,0.422166,70.648121,9.239443,98.124842,None,4.57,36.0,17.987500
1361,1362,00-0030564,2020,WR,nfl,17.987500,17.902667,19.373208,19.799917,-1.375333,16.0,78.0,4.722254,5.486062,10.923625,0.921250,-0.047574,0.062500,10.000000,8.793750,1.0,4.513043,0.982542,0.335684,NaN,NaN,3.142847,None,0,0,0.000000,0,0,0,None,28.238193,21.237509,14,2.238193,0.5580,0.5417,67.88,606.0,25.62,10.0,1,1,1.0,0.393022,20.238193,NaN,NaN,1.0,1.0,0.422166,70.648121,9.239443,98.124842,None,4.57,36.0,14.720000
1362,1363,00-0030564,2021,WR,nfl,14.720000,17.987500,17.945083,18.911306,-1.428125,10.0,94.0,2.034367,3.283141,5.349749,0.822000,-0.099250,0.000000,6.400000,8.937500,NaN,3.333333,0.725888,0.662156,NaN,NaN,2.829418,None,2,1,0.048529,0,0,0,None,29.237509,21.237509,14,3.237509,0.5655,0.5717,65.72,669.0,26.41,19.0,0,1,1.0,0.393022,20.238193,NaN,NaN,1.0,1.0,0.422166,70.648121,9.239443,98.124842,None,4.57,36.0,16.855556
1363,1364,00-0030564,2022,WR,nfl,16.855556,14.720000,16.353750,16.870056,-1.5

In [540]:
missing_pct = (
    eng_feat_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                      feature  missing_pct
0           height_weight_bmi     1.000000
1    injury_designation_count     1.000000
2    catch_pct_above_expected     1.000000
3                ryoe_per_att     0.936639
4                        cpoe     0.934670
5              separation_avg     0.797566
6                 sparq_score     0.745660
7                breakout_age     0.743154
8               vertical_jump     0.627707
9                 speed_score     0.610703
10                 forty_yard     0.609272
11    relative_athletic_score     0.558797
12       college_tds_per_game     0.535887
13     college_yards_per_game     0.535887
14          fantasy_ppg_trend     0.513514
15            yards_per_carry     0.431359
16           dominator_rating     0.394308
17    college_conference_tier     0.361196
18    fantasy_ppg_next_season     0.341328
19             snap_pct_trend     0.309289
20    fantasy_ppg_last_season     0.298729
21      draft_pick_normalized     0.298729
22         

In [14]:
injury_df

,id,player_id,season,week,team,report_status,practice_status,primary_injury
0,1,00-0039851,2024,7,NE,None,Full Participation in Practice,None
1,2,00-0039851,2024,8,NE,None,Full Participation in Practice,None
2,3,00-0039851,2024,9,NE,Questionable,Limited Participation in Practice,None
3,4,00-0039851,2024,18,NE,Questionable,Limited Participation in Practice,None
4,5,00-0039910,2024,8,WAS,Questionable,Limited Participation in Practice,None
...,...,...,...,...,...,...,...,...
44790,44791,00-0026512,2016,17,WAS,Questionable,Limited Participation in Practice,None
44791,44792,00-0026512,2017,3,TB,Doubtful,Did Not Participate In Practice,None
44792,44793,00-0026512,2017,4,TB,None,Full Participation in Practice,None
44793,44794,00-0026512,2017,8,TB,Questionable,Full Participation in Practice,None
